# 🔬 AI Research Paper Explainer
## Full System · GPU · Streamlit UI · Google Colab

| Split | Papers | Purpose |
|-------|--------|---------|
| **Train/Test** | Attention Is All You Need · BERT | Build + tune RAG |
| **Validation** | RAG (Lewis et al.) | Unseen test |

**Run order:** Cell 1 → 2 → 3 → GPU → 4 → 5 → 6–16 → Cell UI

> No uploads needed. Cell 2 writes all files automatically.
> No GPU needed for mock LLM. GPU enables TinyLlama real answers.

---
## Cell 1 — Install packages

In [ ]:
import subprocess, sys
pkgs = [
    'PyMuPDF>=1.23.0','sentence-transformers>=2.7.0',
    'faiss-cpu>=1.8.0','transformers>=4.40.0',
    'accelerate>=0.28.0','huggingface_hub>=0.22.0',
    'nltk>=3.8.1','numpy>=1.26.0','pandas>=2.2.0',
    'scikit-learn>=1.4.0','requests>=2.31.0',
    'streamlit>=1.33.0','plotly>=5.20.0',
    'tabulate>=0.9.0','tqdm>=4.66.0','pyngrok>=7.0.0',
]
r = subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, capture_output=True, text=True)
print('✅ Packages installed' if r.returncode==0 else f'Error: {r.stderr[-300:]}')
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ NLTK ready')

---
## Cell 2 — Write all project source files (self-contained)

In [ ]:
import os, sys
PROJECT = '/content/rag_project'
os.makedirs(PROJECT, exist_ok=True)
os.makedirs(PROJECT+'/data', exist_ok=True)
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

open('dataset.py','w').write('"""\ndataset.py\n==========\nDownloads and caches PDFs from arXiv.\nExtracts clean text page-by-page using PyMuPDF.\n\nDataset split\n-------------\n  Train / Test : Attention Is All You Need  (1706.03762)\n                 BERT                        (1810.04805)\n  Validation   : RAG                         (2005.11401)\n  → RAG paper must NOT be loaded until validation phase.\n"""\n\nimport os\nimport re\nimport logging\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import List, Optional\n\nimport requests\nimport fitz  # PyMuPDF\n\nlogger = logging.getLogger(__name__)\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\n\n# ---------------------------------------------------------------------------\n# Registry\n# ---------------------------------------------------------------------------\nPAPERS = {\n    "attention": {\n        "title":    "Attention Is All You Need",\n        "authors":  "Vaswani et al.",\n        "year":     2017,\n        "url":      "https://arxiv.org/pdf/1706.03762",\n        "filename": "attention.pdf",\n        "split":    "train_test",\n    },\n    "bert": {\n        "title":    "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",\n        "authors":  "Devlin et al.",\n        "year":     2018,\n        "url":      "https://arxiv.org/pdf/1810.04805",\n        "filename": "bert.pdf",\n        "split":    "train_test",\n    },\n    "rag": {\n        "title":    "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",\n        "authors":  "Lewis et al.",\n        "year":     2020,\n        "url":      "https://arxiv.org/pdf/2005.11401",\n        "filename": "rag.pdf",\n        "split":    "validation",\n    },\n}\n\nDATA_DIR = "data"\n\n\n# ---------------------------------------------------------------------------\n# Data model\n# ---------------------------------------------------------------------------\n@dataclass\nclass Page:\n    number: int\n    text: str\n    headings: List[str] = field(default_factory=list)\n\n\n@dataclass\nclass PaperDoc:\n    paper_id: str\n    title: str\n    authors: str\n    year: int\n    split: str\n    pages: List[Page] = field(default_factory=list)\n\n    @property\n    def full_text(self) -> str:\n        return " ".join(p.text for p in self.pages)\n\n    @property\n    def word_count(self) -> int:\n        return len(self.full_text.split())\n\n    @property\n    def page_count(self) -> int:\n        return len(self.pages)\n\n    def __repr__(self):\n        return (f"PaperDoc(id={self.paper_id!r}, pages={self.page_count}, "\n                f"words={self.word_count:,})")\n\n\n# ---------------------------------------------------------------------------\n# Download\n# ---------------------------------------------------------------------------\ndef _download(url: str, dest: Path) -> None:\n    headers = {"User-Agent": "Mozilla/5.0 (research-bot/2.0)"}\n    logger.info(f"Downloading {url} ...")\n    resp = requests.get(url, headers=headers, timeout=120, stream=True)\n    resp.raise_for_status()\n    dest.write_bytes(resp.content)\n    logger.info(f"Saved → {dest}  ({dest.stat().st_size / 1024:.1f} KB)")\n\n\n# ---------------------------------------------------------------------------\n# PDF extraction\n# ---------------------------------------------------------------------------\n_HEADING_RE = re.compile(\n    r"^(?:\\d+\\.?\\s+)?[A-Z][A-Z\\s\\-:]{3,60}$", re.MULTILINE\n)\n_KNOWN_SECTIONS = re.compile(\n    r"\\b(abstract|introduction|related work|background|methodology|method|"\n    r"experiment|result|discussion|conclusion|reference|appendix|"\n    r"attention|transformer|pre.?training|fine.?tuning|retrieval|generation)\\b",\n    re.IGNORECASE,\n)\n\n\ndef _extract_pdf(pdf_path: str) -> List[Page]:\n    doc = fitz.open(pdf_path)\n    pages: List[Page] = []\n    for i, page in enumerate(doc):\n        raw_text = page.get_text("text")\n        # clean\n        text = re.sub(r"\\s+", " ", raw_text).strip()\n        text = re.sub(r"(\\w)-\\s+(\\w)", r"\\1\\2", text)  # de-hyphenate\n        # detect headings\n        headings = []\n        for line in raw_text.splitlines():\n            line = line.strip()\n            if _KNOWN_SECTIONS.match(line) or (\n                len(line) < 80 and line.isupper() and len(line.split()) >= 2\n            ):\n                headings.append(line)\n        pages.append(Page(number=i + 1, text=text, headings=headings))\n    doc.close()\n    return pages\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\ndef load_paper(paper_id: str, data_dir: str = DATA_DIR) -> PaperDoc:\n    """Download (if needed) and return a PaperDoc."""\n    if paper_id not in PAPERS:\n        raise ValueError(f"Unknown paper {paper_id!r}. Choose from {list(PAPERS)}")\n    meta = PAPERS[paper_id]\n    os.makedirs(data_dir, exist_ok=True)\n    pdf_path = Path(data_dir) / meta["filename"]\n\n    if not pdf_path.exists():\n        _download(meta["url"], pdf_path)\n    else:\n        logger.info(f"Using cached PDF: {pdf_path}")\n\n    pages = _extract_pdf(str(pdf_path))\n    doc = PaperDoc(\n        paper_id=paper_id,\n        title=meta["title"],\n        authors=meta["authors"],\n        year=meta["year"],\n        split=meta["split"],\n        pages=pages,\n    )\n    logger.info(f"Loaded: {doc}")\n    return doc\n\n\ndef load_train_test(data_dir: str = DATA_DIR):\n    """Load Attention + BERT (train/test papers)."""\n    return {k: load_paper(k, data_dir) for k in ("attention", "bert")}\n\n\ndef load_validation(data_dir: str = DATA_DIR) -> PaperDoc:\n    """Load RAG paper (validation only — call after system is tuned)."""\n    return load_paper("rag", data_dir)\n\n\nif __name__ == "__main__":\n    docs = load_train_test()\n    for doc in docs.values():\n        print(doc)\n')
print('  wrote dataset.py')
open('chunking.py','w').write('"""\nchunking.py\n===========\nFive modular, configurable chunking strategies.\n\nStrategies\n----------\n1. fixed        – non-overlapping fixed-token windows\n2. sentence     – sentence-boundary-aware accumulation\n3. dynamic      – adaptive 200–800 token windows (flush at natural boundary)\n4. overlapping  – fixed window with configurable stride/overlap\n5. heading      – splits at detected section headings, sub-chunks large sections\n\nAll strategies return List[Chunk] with rich metadata.\n"""\n\nimport re\nimport logging\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional\n\nimport nltk\n\ntry:\n    nltk.data.find("tokenizers/punkt_tab")\nexcept LookupError:\n    nltk.download("punkt_tab", quiet=True)\ntry:\n    nltk.data.find("tokenizers/punkt")\nexcept LookupError:\n    nltk.download("punkt", quiet=True)\n\nfrom nltk.tokenize import sent_tokenize\n\nlogger = logging.getLogger(__name__)\n\n# ---------------------------------------------------------------------------\n# Data model\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Chunk:\n    chunk_id: str\n    paper_id: str\n    strategy: str\n    text: str\n    token_count: int\n    index: int\n    section: Optional[str] = None\n    page_hint: int = -1\n    metadata: dict = field(default_factory=dict)\n\n    def __len__(self):\n        return self.token_count\n\n    def __repr__(self):\n        return (f"Chunk(id={self.chunk_id!r}, strategy={self.strategy!r}, "\n                f"tokens={self.token_count}, section={self.section!r})")\n\n\ndef _tok(text: str) -> int:\n    """Approximate token count (whitespace split — fast & consistent)."""\n    return len(text.split())\n\n\ndef _make(paper_id: str, strategy: str, idx: int, text: str,\n          section: Optional[str] = None, page_hint: int = -1) -> Chunk:\n    text = text.strip()\n    return Chunk(\n        chunk_id=f"{paper_id}_{strategy}_{idx:04d}",\n        paper_id=paper_id,\n        strategy=strategy,\n        text=text,\n        token_count=_tok(text),\n        index=idx,\n        section=section,\n        page_hint=page_hint,\n    )\n\n\n# ---------------------------------------------------------------------------\n# Strategy 1 — Fixed-size\n# ---------------------------------------------------------------------------\n\ndef chunk_fixed(text: str, paper_id: str, size: int = 500) -> List[Chunk]:\n    """Split text into non-overlapping windows of exactly `size` tokens."""\n    words = text.split()\n    chunks: List[Chunk] = []\n    for i, start in enumerate(range(0, len(words), size)):\n        window = words[start: start + size]\n        if not window:\n            break\n        chunks.append(_make(paper_id, "fixed", i, " ".join(window)))\n    return [c for c in chunks if c.token_count >= 20]\n\n\n# ---------------------------------------------------------------------------\n# Strategy 2 — Sentence-aware\n# ---------------------------------------------------------------------------\n\ndef chunk_sentence(text: str, paper_id: str, size: int = 500) -> List[Chunk]:\n    """Accumulate sentences until token budget is full, then flush."""\n    sentences = sent_tokenize(text)\n    chunks: List[Chunk] = []\n    buf: List[str] = []\n    buf_tok = 0\n    idx = 0\n\n    for sent in sentences:\n        t = _tok(sent)\n        if buf_tok + t > size and buf:\n            chunks.append(_make(paper_id, "sentence", idx, " ".join(buf)))\n            idx += 1\n            buf, buf_tok = [], 0\n        buf.append(sent)\n        buf_tok += t\n\n    if buf:\n        chunks.append(_make(paper_id, "sentence", idx, " ".join(buf)))\n    return [c for c in chunks if c.token_count >= 20]\n\n\n# ---------------------------------------------------------------------------\n# Strategy 3 — Dynamic (200–800 tokens)\n# ---------------------------------------------------------------------------\n\ndef chunk_dynamic(text: str, paper_id: str,\n                  lo: int = 200, hi: int = 800) -> List[Chunk]:\n    """\n    Accumulate sentences freely between lo and hi tokens.\n    Flush early when we hit lo AND the last sentence ends naturally.\n    """\n    paragraphs = re.split(r"\\n{2,}", text)\n    sentences: List[str] = []\n    for para in paragraphs:\n        sentences.extend(sent_tokenize(para.strip()))\n\n    chunks: List[Chunk] = []\n    buf: List[str] = []\n    buf_tok = 0\n    idx = 0\n\n    for sent in sentences:\n        t = _tok(sent)\n        if buf_tok + t > hi and buf:\n            chunks.append(_make(paper_id, "dynamic", idx, " ".join(buf)))\n            idx += 1\n            buf, buf_tok = [], 0\n\n        buf.append(sent)\n        buf_tok += t\n\n        # Early flush: within target range at a natural boundary\n        if buf_tok >= lo and sent.rstrip().endswith((".", "!", "?")):\n            chunks.append(_make(paper_id, "dynamic", idx, " ".join(buf)))\n            idx += 1\n            buf, buf_tok = [], 0\n\n    if buf:\n        chunks.append(_make(paper_id, "dynamic", idx, " ".join(buf)))\n    return [c for c in chunks if c.token_count >= 20]\n\n\n# ---------------------------------------------------------------------------\n# Strategy 4 — Overlapping\n# ---------------------------------------------------------------------------\n\ndef chunk_overlapping(text: str, paper_id: str,\n                      size: int = 500, overlap: int = 100) -> List[Chunk]:\n    """Fixed windows with stride = size - overlap."""\n    if overlap >= size:\n        raise ValueError("overlap must be < size")\n    words = text.split()\n    stride = size - overlap\n    chunks: List[Chunk] = []\n    idx = 0\n    for start in range(0, len(words), stride):\n        window = words[start: start + size]\n        if not window:\n            break\n        chunks.append(_make(paper_id, "overlapping", idx, " ".join(window)))\n        idx += 1\n    return [c for c in chunks if c.token_count >= 20]\n\n\n# ---------------------------------------------------------------------------\n# Strategy 5 — Heading-aware\n# ---------------------------------------------------------------------------\n\n_SECTION_RE = re.compile(\n    r"\\b(abstract|introduction|related work|background|method(?:ology)?|"\n    r"approach|experiment|result|discussion|conclusion|reference|appendix|"\n    r"attention mechanism|transformer|pre.?training|fine.?tuning|"\n    r"retrieval|generation|model|architecture|dataset|evaluation|"\n    r"analysis|limitation|future work)\\b",\n    re.IGNORECASE,\n)\n\n\ndef _detect_section_positions(text: str) -> List[int]:\n    """Return character positions where new sections likely begin."""\n    positions = []\n    for m in _SECTION_RE.finditer(text):\n        line_start = text.rfind("\\n", 0, m.start()) + 1\n        prefix = text[line_start: m.start()].strip()\n        if len(prefix) < 12:\n            positions.append(line_start)\n    return sorted(set(positions))\n\n\ndef chunk_heading(text: str, paper_id: str, max_size: int = 800) -> List[Chunk]:\n    """\n    Split text at detected section headings.\n    Large sections are sub-chunked with sentence-aware strategy.\n    """\n    positions = _detect_section_positions(text)\n\n    if not positions:\n        # Fall back to sentence chunking if no headings found\n        logger.debug(f"[{paper_id}] No headings detected — falling back to sentence chunking")\n        return chunk_sentence(text, paper_id, max_size)\n\n    # Build sections\n    boundaries = [0] + positions + [len(text)]\n    raw_sections = []\n    for i in range(len(boundaries) - 1):\n        sec = text[boundaries[i]: boundaries[i + 1]].strip()\n        if sec:\n            raw_sections.append(sec)\n\n    chunks: List[Chunk] = []\n    idx = 0\n\n    for sec in raw_sections:\n        # Extract heading hint from first line\n        first_line = sec.split("\\n")[0].strip()[:80]\n        heading = first_line if len(first_line) < 80 else None\n\n        if _tok(sec) <= max_size:\n            c = _make(paper_id, "heading", idx, sec, section=heading)\n            chunks.append(c)\n            idx += 1\n        else:\n            # Sub-chunk large section\n            sub_chunks = chunk_sentence(sec, paper_id, max_size)\n            for sc in sub_chunks:\n                sc.chunk_id = f"{paper_id}_heading_{idx:04d}"\n                sc.strategy = "heading"\n                sc.index = idx\n                sc.section = heading\n                chunks.append(sc)\n                idx += 1\n\n    return [c for c in chunks if c.token_count >= 20]\n\n\n# ---------------------------------------------------------------------------\n# Registry & factory\n# ---------------------------------------------------------------------------\n\nSTRATEGIES = {\n    "fixed":       chunk_fixed,\n    "sentence":    chunk_sentence,\n    "dynamic":     chunk_dynamic,\n    "overlapping": chunk_overlapping,\n    "heading":     chunk_heading,\n}\n\n\ndef get_chunks(text: str, paper_id: str, strategy: str = "dynamic",\n               **kwargs) -> List[Chunk]:\n    """\n    Public factory.  Extra kwargs are forwarded to the strategy function.\n    """\n    if strategy not in STRATEGIES:\n        raise ValueError(f"Unknown strategy {strategy!r}. Choose from {list(STRATEGIES)}")\n    chunks = STRATEGIES[strategy](text, paper_id, **kwargs)\n    logger.info(f"[{paper_id}] strategy={strategy}  chunks={len(chunks)}")\n    return chunks\n\n\n# ---------------------------------------------------------------------------\n# Comparison utility (used by experiments.py)\n# ---------------------------------------------------------------------------\n\ndef compare_strategies(text: str, paper_id: str,\n                        sizes: List[int] = (200, 500, 800)) -> List[dict]:\n    """\n    Run all strategies × sizes and return a comparison table.\n    """\n    rows = []\n    total_words = len(text.split())\n    for strat in STRATEGIES:\n        for sz in sizes:\n            kw = {}\n            if strat == "fixed":        kw = {"size": sz}\n            elif strat == "sentence":   kw = {"size": sz}\n            elif strat == "dynamic":    kw = {"lo": max(50, sz // 4), "hi": sz}\n            elif strat == "overlapping":kw = {"size": sz, "overlap": sz // 5}\n            elif strat == "heading":    kw = {"max_size": sz}\n            try:\n                chunks = STRATEGIES[strat](text, paper_id, **kw)\n                toks = [c.token_count for c in chunks]\n                covered = sum(toks)\n                rows.append({\n                    "strategy":   strat,\n                    "target_size":sz,\n                    "num_chunks": len(chunks),\n                    "avg_tokens": round(sum(toks) / max(len(toks), 1), 1),\n                    "min_tokens": min(toks, default=0),\n                    "max_tokens": max(toks, default=0),\n                    "coverage":   round(covered / max(total_words, 1), 3),\n                })\n            except Exception as e:\n                rows.append({"strategy": strat, "target_size": sz, "error": str(e)})\n    return rows\n\n\nif __name__ == "__main__":\n    sample = (\n        "The Transformer model relies entirely on attention mechanisms. "\n        "Multi-head attention allows the model to attend to multiple representation subspaces. "\n        "The encoder maps input to continuous representations. "\n        "The decoder generates outputs one token at a time. "\n        "Positional encodings are added to give the model a sense of order. "\n        "Results show improvements over RNN-based sequence models on translation tasks. "\n    ) * 30\n\n    for row in compare_strategies(sample, "test", sizes=[200, 500]):\n        if "error" not in row:\n            print(f"  {row[\'strategy\']:12s} sz={row[\'target_size\']:4d}  "\n                  f"chunks={row[\'num_chunks\']:3d}  avg={row[\'avg_tokens\']:6.1f}")\n')
print('  wrote chunking.py')
open('retrieval.py','w').write('"""\nretrieval.py\n============\nEmbedding generation (sentence-transformers all-MiniLM-L6-v2),\nFAISS flat inner-product index, and optional cross-encoder reranking.\n\nDesign choices\n--------------\n* all-MiniLM-L6-v2  →  384-d, L2-normalised → cosine via inner product\n* IndexFlatIP       →  exact search, no approximation errors\n* Source filtering  →  retrieve only from the queried paper\n* Cross-encoder     →  ms-marco-MiniLM-L-6-v2 for precision boost\n"""\n\nimport logging\nimport pickle\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import List, Optional, Tuple\n\nimport numpy as np\n\nlogger = logging.getLogger(__name__)\n\nEMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"\nRERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"\n\n\n# ---------------------------------------------------------------------------\n# Data model\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass SearchResult:\n    chunk: object          # Chunk from chunking.py\n    bi_score: float        # FAISS cosine score\n    rerank_score: Optional[float] = None\n    rank: int = 0\n\n    @property\n    def score(self) -> float:\n        return self.rerank_score if self.rerank_score is not None else self.bi_score\n\n\n# ---------------------------------------------------------------------------\n# Embedder\n# ---------------------------------------------------------------------------\n\nclass Embedder:\n    """Wraps SentenceTransformer with L2-normalised output."""\n\n    def __init__(self, model_name: str = EMBED_MODEL, device: str = "cpu"):\n        from sentence_transformers import SentenceTransformer\n        logger.info(f"Loading embedding model: {model_name}")\n        self._model = SentenceTransformer(model_name, device=device)\n        self.dim = self._model.get_sentence_embedding_dimension()\n        logger.info(f"Embedding dim: {self.dim}")\n\n    def encode(self, texts: List[str], batch_size: int = 64,\n               show_progress: bool = False) -> np.ndarray:\n        vecs = self._model.encode(\n            texts,\n            batch_size=batch_size,\n            show_progress_bar=show_progress,\n            normalize_embeddings=True,\n            convert_to_numpy=True,\n        )\n        return vecs.astype(np.float32)\n\n    def encode_one(self, text: str) -> np.ndarray:\n        return self.encode([text])[0]\n\n\n# ---------------------------------------------------------------------------\n# FAISS index wrapper\n# ---------------------------------------------------------------------------\n\nclass VectorIndex:\n    """\n    Maintains a FAISS IndexFlatIP (cosine on normalised vectors).\n    Supports per-paper source filtering.\n    """\n\n    def __init__(self, dim: int):\n        import faiss\n        self.dim = dim\n        self._index = faiss.IndexFlatIP(dim)\n        self._chunks: List[object] = []          # parallel array to FAISS positions\n        self._source_map: List[str] = []         # paper_id for each position\n\n    @property\n    def size(self) -> int:\n        return len(self._chunks)\n\n    def add(self, embeddings: np.ndarray, chunks: List[object]) -> None:\n        assert embeddings.shape[0] == len(chunks), "embedding/chunk count mismatch"\n        self._index.add(embeddings)\n        for chunk in chunks:\n            self._chunks.append(chunk)\n            self._source_map.append(chunk.paper_id)\n        logger.info(f"VectorIndex: {self.size} vectors total")\n\n    def search(self, query_vec: np.ndarray, k: int,\n               source_filter: Optional[str] = None) -> List[Tuple[float, object]]:\n        """Return up to k (score, chunk) pairs, optionally filtered by paper."""\n        fetch_k = k if source_filter is None else min(k * 12, self.size)\n        fetch_k = max(fetch_k, 1)\n        qv = query_vec.reshape(1, -1).astype(np.float32)\n        scores, ids = self._index.search(qv, min(fetch_k, self.size))\n\n        results = []\n        for score, idx in zip(scores[0], ids[0]):\n            if idx < 0:\n                continue\n            if source_filter and self._source_map[idx] != source_filter:\n                continue\n            results.append((float(score), self._chunks[idx]))\n            if len(results) >= k:\n                break\n        return results\n\n    def reset(self) -> None:\n        import faiss\n        self._index = faiss.IndexFlatIP(self.dim)\n        self._chunks.clear()\n        self._source_map.clear()\n\n    def save(self, path: str) -> None:\n        import faiss\n        p = Path(path)\n        p.parent.mkdir(parents=True, exist_ok=True)\n        faiss.write_index(self._index, str(p) + ".faiss")\n        with open(str(p) + ".meta.pkl", "wb") as f:\n            pickle.dump((self._chunks, self._source_map), f)\n\n    @classmethod\n    def load(cls, path: str, dim: int) -> "VectorIndex":\n        import faiss\n        obj = cls.__new__(cls)\n        obj.dim = dim\n        obj._index = faiss.read_index(str(path) + ".faiss")\n        with open(str(path) + ".meta.pkl", "rb") as f:\n            obj._chunks, obj._source_map = pickle.load(f)\n        return obj\n\n\n# ---------------------------------------------------------------------------\n# Cross-encoder reranker (optional)\n# ---------------------------------------------------------------------------\n\nclass Reranker:\n    def __init__(self, model_name: str = RERANK_MODEL):\n        from sentence_transformers import CrossEncoder\n        logger.info(f"Loading reranker: {model_name}")\n        self._model = CrossEncoder(model_name)\n\n    def rerank(self, query: str, results: List[SearchResult],\n               top_n: Optional[int] = None) -> List[SearchResult]:\n        if not results:\n            return results\n        pairs = [(query, r.chunk.text) for r in results]\n        scores = self._model.predict(pairs)\n        for r, s in zip(results, scores):\n            r.rerank_score = float(s)\n        results.sort(key=lambda x: x.rerank_score, reverse=True)  # type: ignore\n        if top_n:\n            results = results[:top_n]\n        for i, r in enumerate(results):\n            r.rank = i + 1\n        return results\n\n\n# ---------------------------------------------------------------------------\n# High-level retrieval pipeline\n# ---------------------------------------------------------------------------\n\nclass RetrievalPipeline:\n    """\n    Public interface used by pipeline.py.\n\n    Usage\n    -----\n    rp = RetrievalPipeline(use_reranker=False)\n    rp.index_chunks(chunks)\n    results = rp.retrieve("How does attention work?", paper_id="attention", k=5)\n    """\n\n    def __init__(self, embed_model: str = EMBED_MODEL,\n                 use_reranker: bool = False,\n                 device: str = "cpu"):\n        self.embedder = Embedder(embed_model, device=device)\n        self._index: Optional[VectorIndex] = None\n        self._reranker: Optional[Reranker] = None\n        if use_reranker:\n            try:\n                self._reranker = Reranker()\n            except Exception as e:\n                logger.warning(f"Reranker unavailable: {e}. Proceeding without.")\n\n    def index_chunks(self, chunks: List[object],\n                     batch_size: int = 128, show_progress: bool = True) -> None:\n        """Embed and index all chunks. Can be called multiple times to add papers."""\n        if self._index is None:\n            self._index = VectorIndex(dim=self.embedder.dim)\n        texts = [c.text for c in chunks]\n        logger.info(f"Embedding {len(texts)} chunks ...")\n        vecs = self.embedder.encode(texts, batch_size=batch_size,\n                                    show_progress=show_progress)\n        self._index.add(vecs, chunks)\n\n    def retrieve(self, query: str, k: int = 5,\n                 paper_id: Optional[str] = None) -> List[SearchResult]:\n        """Retrieve top-k chunks for a query, optionally from one paper only."""\n        if self._index is None or self._index.size == 0:\n            raise RuntimeError("Index is empty. Call index_chunks() first.")\n        q_vec = self.embedder.encode_one(query)\n        raw = self._index.search(q_vec, k=k, source_filter=paper_id)\n        results = [\n            SearchResult(chunk=chunk, bi_score=score, rank=i + 1)\n            for i, (score, chunk) in enumerate(raw)\n        ]\n        if self._reranker and results:\n            results = self._reranker.rerank(query, results, top_n=k)\n        return results\n\n    def retrieve_multi(self, query: str, paper_ids: List[str],\n                       k_per_paper: int = 3) -> dict:\n        """Retrieve top-k chunks separately for each paper."""\n        return {\n            pid: self.retrieve(query, k=k_per_paper, paper_id=pid)\n            for pid in paper_ids\n        }\n\n    # ── Metric helpers ────────────────────────────────────────────────────\n    def context_relevance(self, query: str,\n                          results: List[SearchResult]) -> float:\n        """Average cosine sim between query and retrieved chunks."""\n        if not results:\n            return 0.0\n        q_vec = self.embedder.encode_one(query)\n        scores = []\n        for r in results:\n            c_vec = self.embedder.encode_one(r.chunk.text)\n            scores.append(float(np.dot(q_vec, c_vec)))\n        return float(np.mean(scores))\n\n    def reset_index(self) -> None:\n        if self._index:\n            self._index.reset()\n\n\n# ---------------------------------------------------------------------------\n# Standalone metrics\n# ---------------------------------------------------------------------------\n\ndef recall_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:\n    top = retrieved_ids[:k]\n    return len(set(top) & relevant_ids) / max(len(relevant_ids), 1)\n\n\ndef precision_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:\n    top = retrieved_ids[:k]\n    return sum(1 for r in top if r in relevant_ids) / max(k, 1)\n\n\nif __name__ == "__main__":\n    from chunking import chunk_dynamic\n    sample = ("The Transformer uses self-attention. " * 30)\n    chunks = chunk_dynamic(sample, "test")\n    rp = RetrievalPipeline(use_reranker=False)\n    rp.index_chunks(chunks)\n    results = rp.retrieve("How does attention work?", k=3)\n    for r in results:\n        print(f"  rank={r.rank}  score={r.bi_score:.4f}  tokens={r.chunk.token_count}")\n')
print('  wrote retrieval.py')
open('llm.py','w').write('"""\nllm.py\n======\nLLM backends (TinyLlama / Mock) + structured answer generation.\n\nMandatory output format\n-----------------------\n  Core Idea:\n  Methodology:\n  Key Results:\n  Limitations:\n  ELI12 Explanation:\n\n  → If section is absent: "Not specified in the paper"\n\nGrounding rule\n--------------\n  The LLM sees ONLY retrieved context chunks — no external knowledge.\n  The strict prompt explicitly forbids using outside information.\n"""\n\nimport re\nimport time\nimport logging\nimport warnings\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional\n\n# Suppress noisy transformers internal warnings\nwarnings.filterwarnings("ignore", message=".*__path__.*")\nwarnings.filterwarnings("ignore", message=".*torch_dtype.*deprecated.*")\nwarnings.filterwarnings("ignore", message=".*use_container_width.*")\n\nlogger = logging.getLogger(__name__)\n\nNOT_SPECIFIED = "Not specified in the paper"\nSECTIONS = ["Core Idea", "Methodology", "Key Results", "Limitations", "ELI12 Explanation"]\n\n# ---------------------------------------------------------------------------\n# Prompt templates\n# ---------------------------------------------------------------------------\n\nSTRICT_PROMPT = """\\\nYou are an expert AI research analyst.\nAnswer the question using ONLY the provided context passages.\nDo NOT use any external knowledge. If information is missing, write:\n  "Not specified in the paper"\n\nCONTEXT PASSAGES:\n{context}\n\nQUESTION: {question}\n\nRespond in this EXACT format (all five sections required):\n\nCore Idea:\n[Explain the central contribution using ONLY the context]\n\nMethodology:\n[Describe the methods/techniques using ONLY the context]\n\nKey Results:\n[State the key results/findings from the context]\n\nLimitations:\n[List known limitations from the context]\n\nELI12 Explanation:\n[Explain to a curious 12-year-old using simple words, based on context]\n\nANSWER:"""\n\nOPEN_PROMPT = """\\\nYou are an expert AI research analyst.\nAnswer the question based on the context, and you may use general knowledge to clarify.\n\nCONTEXT PASSAGES:\n{context}\n\nQUESTION: {question}\n\nRespond in this EXACT format:\n\nCore Idea:\n[Central contribution]\n\nMethodology:\n[Methods and techniques]\n\nKey Results:\n[Key findings]\n\nLimitations:\n[Known limitations]\n\nELI12 Explanation:\n[Simple explanation for a 12-year-old]\n\nANSWER:"""\n\n\ndef build_prompt(question: str, context_chunks: List[str],\n                 style: str = "strict") -> str:\n    ctx = "\\n\\n---\\n\\n".join(\n        f"[Passage {i+1}]:\\n{c}" for i, c in enumerate(context_chunks)\n    )\n    template = STRICT_PROMPT if style == "strict" else OPEN_PROMPT\n    return template.format(context=ctx, question=question)\n\n\n# ---------------------------------------------------------------------------\n# Structured answer\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass StructuredAnswer:\n    core_idea:   str = NOT_SPECIFIED\n    methodology: str = NOT_SPECIFIED\n    key_results: str = NOT_SPECIFIED\n    limitations: str = NOT_SPECIFIED\n    eli12:       str = NOT_SPECIFIED\n    raw_text:    str = ""\n    tokens_used: int = 0\n    latency_s:   float = 0.0\n    model_name:  str = ""\n    prompt_style: str = "strict"\n\n    def to_dict(self) -> dict:\n        return {\n            "Core Idea":         self.core_idea,\n            "Methodology":       self.methodology,\n            "Key Results":       self.key_results,\n            "Limitations":       self.limitations,\n            "ELI12 Explanation": self.eli12,\n        }\n\n    def completeness(self) -> float:\n        filled = sum(\n            1 for v in self.to_dict().values()\n            if v and v != NOT_SPECIFIED\n        )\n        return filled / len(SECTIONS)\n\n    def is_complete(self) -> bool:\n        return self.completeness() == 1.0\n\n    def __str__(self):\n        divider = "─" * 52\n        lines = []\n        for k, v in self.to_dict().items():\n            lines.append(f"\\n{divider}")\n            lines.append(f"  {k}")\n            lines.append(divider)\n            lines.append(f"  {v[:400]}")\n        lines.append(f"\\n[completeness={self.completeness():.0%} | "\n                     f"tokens={self.tokens_used} | {self.latency_s*1000:.0f}ms | "\n                     f"model={self.model_name}]")\n        return "\\n".join(lines)\n\n\ndef _parse(raw: str) -> StructuredAnswer:\n    """Parse LLM output into StructuredAnswer."""\n    ans = StructuredAnswer(raw_text=raw)\n    patterns = {\n        "core_idea":   r"Core Idea:\\s*(.*?)(?=Methodology:|Key Results:|Limitations:|ELI12|$)",\n        "methodology": r"Methodology:\\s*(.*?)(?=Key Results:|Limitations:|ELI12|Core Idea:|$)",\n        "key_results": r"Key Results:\\s*(.*?)(?=Limitations:|ELI12|Core Idea:|Methodology:|$)",\n        "limitations": r"Limitations:\\s*(.*?)(?=ELI12|Core Idea:|Methodology:|Key Results:|$)",\n        "eli12":       r"ELI12 Explanation:\\s*(.*?)(?=Core Idea:|Methodology:|Key Results:|Limitations:|$)",\n    }\n    for attr, pat in patterns.items():\n        m = re.search(pat, raw, re.DOTALL | re.IGNORECASE)\n        if m:\n            val = re.sub(r"\\s+", " ", m.group(1)).strip()\n            if val and len(val) > 5 and val.lower() != "not specified in the paper":\n                setattr(ans, attr, val)\n            elif val and val.lower() == "not specified in the paper":\n                setattr(ans, attr, NOT_SPECIFIED)\n    return ans\n\n\n# ---------------------------------------------------------------------------\n# Backends\n# ---------------------------------------------------------------------------\n\nclass _MockBackend:\n    """Deterministic mock — fast, no GPU. Grounds answer in first context passage."""\n    name = "mock-llm"\n\n    def generate(self, prompt: str, max_new_tokens: int = 512) -> tuple:\n        # Extract first passage to ground the response\n        m = re.search(r"\\[Passage 1\\]:\\n(.*?)(?:\\n---|\\Z)", prompt, re.DOTALL)\n        excerpt = m.group(1).strip()[:200] if m else "the paper"\n\n        text = (\n            f"Core Idea:\\n"\n            f"Based on the retrieved context, the paper\'s core contribution is: {excerpt[:160]}\\n\\n"\n            f"Methodology:\\n"\n            f"The methodology described in the context involves systematic experiments "\n            f"and ablation studies using standard benchmarks as detailed in the passages.\\n\\n"\n            f"Key Results:\\n"\n            f"The context reports competitive improvements over prior baselines on the evaluated tasks. "\n            f"Specific metrics are described in the retrieved passages above.\\n\\n"\n            f"Limitations:\\n"\n            f"The context mentions computational cost and data requirements as key limitations.\\n\\n"\n            f"ELI12 Explanation:\\n"\n            f"Imagine a very smart assistant that reads many books to answer questions. "\n            f"This paper teaches the assistant a smarter way to find and use information."\n        )\n        tokens = len(prompt.split()) + len(text.split())\n        return text, tokens\n\n\nclass _TinyLlamaBackend:\n    """\n    TinyLlama-1.1B-Chat — works on GPU (CUDA), Apple Silicon (MPS), and CPU.\n\n    Memory strategy (auto-selected):\n      CUDA GPU  →  float16  ~2 GB VRAM   (fastest)\n      Apple MPS →  float16  ~2 GB RAM    (fast on M1/M2/M3)\n      CPU only  →  int8 quantisation via bitsandbytes if available,\n                   else float32 with low_cpu_mem_usage + max_memory cap\n                   (prevents the 78% freeze on low-RAM machines)\n    """\n    name = "TinyLlama-1.1B-Chat"\n\n    def __init__(self):\n        self._pipe = None\n\n    def _load(self):\n        if self._pipe is not None:\n            return\n\n        import torch\n        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline\n\n        model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"\n        print(f"[LLM] Loading {model_id} ...")\n\n        tok = AutoTokenizer.from_pretrained(model_id)\n\n        # ── Detect best device ──────────────────────────────────────────\n        if torch.cuda.is_available():\n            device_str = "cuda"\n            dtype       = torch.float16\n            load_kwargs = dict(dtype=dtype, low_cpu_mem_usage=True)\n            device_idx  = 0\n            print("[LLM] Using CUDA GPU (float16)")\n\n        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():\n            device_str = "mps"\n            dtype       = torch.float16\n            load_kwargs = dict(dtype=dtype, low_cpu_mem_usage=True)\n            device_idx  = "mps"\n            print("[LLM] Using Apple MPS (float16)")\n\n        else:\n            # ── CPU path — try 8-bit quantisation first ─────────────────\n            device_str = "cpu"\n            device_idx  = -1\n            try:\n                import bitsandbytes  # noqa: F401\n                load_kwargs = dict(\n                    load_in_8bit=True,\n                    low_cpu_mem_usage=True,\n                    device_map="cpu",\n                )\n                print("[LLM] Using CPU + 8-bit quantisation (bitsandbytes) — ~1.1 GB RAM")\n            except ImportError:\n                # Fall back to float32 with a memory cap so it doesn\'t stall\n                import psutil\n                available_gb = psutil.virtual_memory().available / 1e9\n                # Reserve 2 GB for OS + Streamlit; cap model at what remains\n                model_ram = max(1.5, available_gb - 2.0)\n                load_kwargs = dict(\n                    dtype=torch.float32,\n                    low_cpu_mem_usage=True,\n                    max_memory={0: f"{model_ram:.0f}GB", "cpu": f"{model_ram:.0f}GB"},\n                )\n                print(f"[LLM] Using CPU float32 — available RAM: {available_gb:.1f} GB")\n                print("[LLM] TIP: run  pip install bitsandbytes psutil  for faster loading")\n\n        # ── Load model ──────────────────────────────────────────────────\n        try:\n            mdl = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)\n        except Exception as e:\n            print(f"[LLM] Model load failed: {e}")\n            print("[LLM] Falling back to mock backend for this session.")\n            # Degrade gracefully — swap to mock so the app keeps running\n            self._pipe = _MockBackend()\n            self.name  = "mock-llm (TinyLlama fallback)"\n            return\n\n        # ── Build pipeline ──────────────────────────────────────────────\n        pipe_kwargs = dict(model=mdl, tokenizer=tok)\n        if device_idx != "mps" and "device_map" not in load_kwargs:\n            pipe_kwargs["device"] = device_idx\n\n        self._pipe = pipeline("text-generation", **pipe_kwargs)\n        print(f"[LLM] TinyLlama ready on {device_str} ✓")\n\n    def generate(self, prompt: str, max_new_tokens: int = 512) -> tuple:\n        self._load()\n        # Graceful fallback: if _pipe was replaced with MockBackend after OOM\n        if isinstance(self._pipe, _MockBackend):\n            return self._pipe.generate(prompt, max_new_tokens)\n\n        out = self._pipe(\n            prompt,\n            max_new_tokens=max_new_tokens,\n            do_sample=True,\n            temperature=0.1,\n            pad_token_id=self._pipe.tokenizer.eos_token_id,\n            repetition_penalty=1.1,\n        )\n        text   = out[0]["generated_text"][len(prompt):]\n        tokens = len(prompt.split()) + len(text.split())\n        return text, tokens\n\n\n_BACKENDS = {\n    "mock":      _MockBackend,\n    "tinyllama": _TinyLlamaBackend,\n}\n\n\n# ---------------------------------------------------------------------------\n# LLM engine\n# ---------------------------------------------------------------------------\n\nclass LLMEngine:\n    """\n    Public interface.\n\n    Usage\n    -----\n    engine = LLMEngine(backend="mock")   # or "tinyllama"\n    ans = engine.answer("What is BERT?", context_chunks, style="strict")\n    """\n\n    def __init__(self, backend: str = "mock"):\n        if backend not in _BACKENDS:\n            raise ValueError(f"Unknown backend {backend!r}. Choose from {list(_BACKENDS)}")\n        self._backend = _BACKENDS[backend]()\n        logger.info(f"LLMEngine: backend={self._backend.name}")\n\n    def answer(self, question: str, context_chunks: List[str],\n               style: str = "strict", max_new_tokens: int = 512) -> StructuredAnswer:\n        """\n        Build prompt → generate → parse → return StructuredAnswer.\n        context_chunks must contain ONLY retrieved context (no external text).\n        """\n        prompt = build_prompt(question, context_chunks, style=style)\n        t0 = time.time()\n        raw, tokens = self._backend.generate(prompt, max_new_tokens=max_new_tokens)\n        latency = time.time() - t0\n\n        ans = _parse(raw)\n        ans.raw_text = raw\n        ans.tokens_used = tokens\n        ans.latency_s = latency\n        ans.model_name = self._backend.name\n        ans.prompt_style = style\n\n        logger.info(f"Answer: completeness={ans.completeness():.0%}  "\n                    f"tokens={tokens}  latency={latency*1000:.0f}ms")\n        return ans\n\n    @property\n    def backend_name(self) -> str:\n        return self._backend.name\n\n\nif __name__ == "__main__":\n    engine = LLMEngine(backend="mock")\n    ctx = [\n        "The Transformer uses multi-head self-attention instead of recurrence.",\n        "Results show BLEU 28.4 on WMT 2014 English-to-German translation.",\n        "One limitation is the quadratic O(n²) attention complexity.",\n    ]\n    ans = engine.answer("What is the Transformer?", ctx)\n    print(ans)\n')
print('  wrote llm.py')
open('hallucination.py','w').write('"""\nhallucination.py\n================\nThree-layer hallucination detection.\n\nLayer 1 — Embedding similarity\n    Cosine sim between answer and context.  Low → likely hallucinated.\n\nLayer 2 — Keyword claim verification\n    Sentence-level keyword overlap between each answer sentence and context.\n    Flags sentences with overlap < threshold.\n\nLayer 3 — LLM self-check\n    Prompts the LLM to rate its own grounding (0-1 score).\n    Falls back to heuristic if LLM unavailable.\n\nComposite score\n    weighted_avg(1-embed_sim, 1-claim_support, 1-llm_support)\n    → 0 = fully grounded, 1 = fully hallucinated\n"""\n\nimport re\nimport logging\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional, Tuple\n\nimport numpy as np\n\nlogger = logging.getLogger(__name__)\n\n# Weights for composite score\nW_EMBED = 0.35\nW_CLAIM = 0.45\nW_LLM   = 0.20\n\nGROUNDED_THRESH     = 0.65   # grounding ≥ this → GROUNDED\nPARTIAL_THRESH      = 0.40   # grounding ≥ this → PARTIAL\n\n\n# ---------------------------------------------------------------------------\n# Data models\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass ClaimCheck:\n    sentence: str\n    is_supported: bool\n    overlap_ratio: float\n    best_evidence: str = ""\n\n\n@dataclass\nclass HallucinationReport:\n    verdict: str                        # GROUNDED | PARTIAL | HALLUCINATED\n    score: float                        # 0 = grounded, 1 = hallucinated\n    grounding_score: float              # 1 - score\n    embed_similarity: float\n    claim_support_rate: float\n    llm_support_score: float\n    unsupported_claims: List[str] = field(default_factory=list)\n    claim_checks: List[ClaimCheck] = field(default_factory=list)\n\n    def to_dict(self) -> dict:\n        return {\n            "verdict":              self.verdict,\n            "hallucination_score":  round(self.score, 4),\n            "grounding_score":      round(self.grounding_score, 4),\n            "embed_similarity":     round(self.embed_similarity, 4),\n            "claim_support_rate":   round(self.claim_support_rate, 4),\n            "llm_support_score":    round(self.llm_support_score, 4),\n            "unsupported_count":    len(self.unsupported_claims),\n            "unsupported_claims":   self.unsupported_claims[:3],\n        }\n\n    def summary(self) -> str:\n        lines = [\n            f"Verdict             : {self.verdict}",\n            f"Hallucination score : {self.score:.3f}  (0=grounded, 1=hallucinated)",\n            f"Embedding similarity: {self.embed_similarity:.3f}",\n            f"Claim support rate  : {self.claim_support_rate:.3f}",\n            f"LLM support score   : {self.llm_support_score:.3f}",\n            f"Unsupported claims  : {len(self.unsupported_claims)}",\n        ]\n        for c in self.unsupported_claims[:2]:\n            lines.append(f"  ✗ {c[:110]}")\n        return "\\n".join(lines)\n\n\n# ---------------------------------------------------------------------------\n# Layer 1 — Embedding similarity\n# ---------------------------------------------------------------------------\n\ndef _embed_similarity(answer_text: str, context_text: str,\n                       embedder) -> float:\n    """Cosine similarity between answer and joined context (L2-normalised)."""\n    a_vec = embedder.encode_one(answer_text)\n    c_vec = embedder.encode_one(context_text)\n    sim = float(np.dot(a_vec, c_vec))\n    return max(0.0, min(1.0, sim))\n\n\n# ---------------------------------------------------------------------------\n# Layer 2 — Keyword claim verification\n# ---------------------------------------------------------------------------\n\ndef _extract_answer_sentences(answer_text: str) -> List[str]:\n    """Pull factual sentences out of the structured answer, skip headers."""\n    # Remove section headers\n    text = re.sub(\n        r"^(Core Idea|Methodology|Key Results|Limitations|ELI12 Explanation"\n        r"|Not specified in the paper)\\s*:?\\s*",\n        "", answer_text, flags=re.MULTILINE\n    )\n    try:\n        from nltk.tokenize import sent_tokenize\n        sents = sent_tokenize(text)\n    except Exception:\n        sents = re.split(r"[.!?]+", text)\n\n    return [\n        s.strip() for s in sents\n        if len(s.split()) >= 6\n        and "not specified" not in s.lower()\n    ][:20]   # cap for efficiency\n\n\ndef _keyword_overlap(claim: str, context: str) -> Tuple[bool, float, str]:\n    """\n    Compute keyword overlap between a claim sentence and context.\n    Returns (is_supported, ratio, best_evidence_snippet).\n    """\n    claim_words = set(re.findall(r"\\b[a-z]{4,}\\b", claim.lower()))\n    ctx_words   = set(re.findall(r"\\b[a-z]{4,}\\b", context.lower()))\n\n    if not claim_words:\n        return True, 1.0, ""\n\n    overlap = claim_words & ctx_words\n    ratio = len(overlap) / len(claim_words)\n    supported = ratio >= 0.30\n\n    # Find best matching context sentence\n    best_evidence = ""\n    best_score = 0.0\n    for sent in re.split(r"[.!?]+", context):\n        sent = sent.strip()\n        if not sent:\n            continue\n        sent_words = set(re.findall(r"\\b[a-z]{4,}\\b", sent.lower()))\n        sc = len(claim_words & sent_words) / max(len(claim_words), 1)\n        if sc > best_score:\n            best_score = sc\n            best_evidence = sent[:120]\n\n    return supported, min(ratio, 1.0), best_evidence\n\n\ndef _claim_support_rate(answer_text: str, context_text: str) -> Tuple[float, List[ClaimCheck]]:\n    """Returns (support_rate, list_of_ClaimCheck)."""\n    sentences = _extract_answer_sentences(answer_text)\n    if not sentences:\n        return 1.0, []\n\n    checks = []\n    for sent in sentences:\n        supported, ratio, evidence = _keyword_overlap(sent, context_text)\n        checks.append(ClaimCheck(\n            sentence=sent,\n            is_supported=supported,\n            overlap_ratio=ratio,\n            best_evidence=evidence,\n        ))\n\n    rate = sum(1 for c in checks if c.is_supported) / len(checks)\n    return rate, checks\n\n\n# ---------------------------------------------------------------------------\n# Layer 3 — LLM self-check\n# ---------------------------------------------------------------------------\n\n_LLM_CHECK_PROMPT = """\\\nYou are a strict fact-checker.\n\nContext:\n{context}\n\nAnswer:\n{answer}\n\nOn a scale from 0.0 to 1.0, how well is the answer supported by the context?\n  1.0 = fully supported\n  0.5 = partially supported\n  0.0 = not supported at all\n\nReply with ONLY a number between 0.0 and 1.0.\nScore:"""\n\n\ndef _llm_support_score(answer_text: str, context_text: str,\n                        llm_engine) -> float:\n    """Use the LLM to self-assess grounding. Falls back to 0.5 on error."""\n    try:\n        prompt = _LLM_CHECK_PROMPT.format(\n            context=context_text[:1200],\n            answer=answer_text[:600],\n        )\n        # Call backend directly for a short generation\n        raw, _ = llm_engine._backend.generate(prompt, max_new_tokens=10)\n        m = re.search(r"([01]?\\.\\d+|[01])", raw)\n        if m:\n            return max(0.0, min(1.0, float(m.group(1))))\n    except Exception as e:\n        logger.debug(f"LLM self-check fallback: {e}")\n\n    # Heuristic fallback: use claim support as proxy\n    return 0.5\n\n\n# ---------------------------------------------------------------------------\n# Main detector\n# ---------------------------------------------------------------------------\n\nclass HallucinationDetector:\n    """\n    Three-layer detector.\n\n    Usage\n    -----\n    detector = HallucinationDetector(embedder, llm_engine)\n    report = detector.detect(answer.raw_text, context_text)\n    """\n\n    def __init__(self, embedder, llm_engine):\n        self._embedder = embedder\n        self._llm = llm_engine\n\n    def detect(self, answer_text: str, context_text: str) -> HallucinationReport:\n        # ── Layer 1 ────────────────────────────────────────────────────────\n        embed_sim = _embed_similarity(answer_text, context_text, self._embedder)\n\n        # ── Layer 2 ────────────────────────────────────────────────────────\n        claim_rate, checks = _claim_support_rate(answer_text, context_text)\n\n        # ── Layer 3 ────────────────────────────────────────────────────────\n        llm_score = _llm_support_score(answer_text, context_text, self._llm)\n\n        # ── Composite ──────────────────────────────────────────────────────\n        grounding = (\n            W_EMBED * embed_sim +\n            W_CLAIM * claim_rate +\n            W_LLM   * llm_score\n        )\n        hall_score = max(0.0, min(1.0, 1.0 - grounding))\n\n        if grounding >= GROUNDED_THRESH:\n            verdict = "GROUNDED"\n        elif grounding >= PARTIAL_THRESH:\n            verdict = "PARTIAL"\n        else:\n            verdict = "HALLUCINATED"\n\n        unsupported = [c.sentence for c in checks if not c.is_supported]\n\n        return HallucinationReport(\n            verdict=verdict,\n            score=round(hall_score, 4),\n            grounding_score=round(grounding, 4),\n            embed_similarity=round(embed_sim, 4),\n            claim_support_rate=round(claim_rate, 4),\n            llm_support_score=round(llm_score, 4),\n            unsupported_claims=unsupported,\n            claim_checks=checks,\n        )\n\n\nif __name__ == "__main__":\n    # Quick smoke-test without real embedder/LLM\n    class _FakeEmb:\n        def encode_one(self, t):\n            import numpy as np\n            v = np.random.randn(384).astype(np.float32)\n            return v / np.linalg.norm(v)\n\n    class _FakeLLM:\n        class _backend:\n            @staticmethod\n            def generate(p, max_new_tokens=10):\n                return "0.8", 2\n\n    det = HallucinationDetector(_FakeEmb(), _FakeLLM())\n    ctx = "The Transformer model uses attention mechanisms. BERT uses MLM pre-training."\n    ans = "The Transformer uses attention. BERT uses masked language modeling."\n    r = det.detect(ans, ctx)\n    print(r.summary())\n')
print('  wrote hallucination.py')
open('evaluation.py','w').write('"""\nevaluation.py\n=============\nAll evaluation metrics used in experiments.\n\nRetrieval metrics\n-----------------\n  recall_at_k, precision_at_k, context_relevance\n\nAnswer quality metrics\n----------------------\n  faithfulness, completeness, hallucination_rate\n\nChunking metrics\n----------------\n  coherence, redundancy, coverage\n\nSystem metrics\n--------------\n  latency_ms, tokens_used\n"""\n\nimport logging\nimport time\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional, Set\n\nimport numpy as np\n\nlogger = logging.getLogger(__name__)\n\n\n# ---------------------------------------------------------------------------\n# Data containers\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass RetrievalMetrics:\n    recall: dict        = field(default_factory=dict)   # {k: float}\n    precision: dict     = field(default_factory=dict)   # {k: float}\n    context_relevance: float = 0.0\n\n    def to_dict(self) -> dict:\n        d = {}\n        for k, v in self.recall.items():\n            d[f"recall@{k}"] = round(v, 4)\n        for k, v in self.precision.items():\n            d[f"precision@{k}"] = round(v, 4)\n        d["context_relevance"] = round(self.context_relevance, 4)\n        return d\n\n\n@dataclass\nclass AnswerMetrics:\n    faithfulness:       float = 0.0\n    completeness:       float = 0.0\n    hallucination_rate: float = 0.0\n\n    def to_dict(self) -> dict:\n        return {\n            "faithfulness":       round(self.faithfulness, 4),\n            "completeness":       round(self.completeness, 4),\n            "hallucination_rate": round(self.hallucination_rate, 4),\n        }\n\n\n@dataclass\nclass ChunkingMetrics:\n    coherence:  float = 0.0   # avg cosine sim of adjacent chunks\n    redundancy: float = 0.0   # avg cosine sim of non-adjacent sample\n    coverage:   float = 0.0   # fraction of source words covered\n\n    def to_dict(self) -> dict:\n        return {\n            "coherence":  round(self.coherence, 4),\n            "redundancy": round(self.redundancy, 4),\n            "coverage":   round(self.coverage, 4),\n        }\n\n\n@dataclass\nclass SystemMetrics:\n    latency_ms:  float = 0.0\n    tokens_used: int   = 0\n    chunks_retrieved: int = 0\n\n    def to_dict(self) -> dict:\n        return {\n            "latency_ms":       round(self.latency_ms, 1),\n            "tokens_used":      self.tokens_used,\n            "chunks_retrieved": self.chunks_retrieved,\n        }\n\n\n@dataclass\nclass EvalRecord:\n    """One complete evaluation record (one query × one config)."""\n    paper_id:    str\n    question:    str\n    strategy:    str\n    target_size: int\n    top_k:       int\n    prompt_style: str\n    retrieval:   RetrievalMetrics  = field(default_factory=RetrievalMetrics)\n    answer:      AnswerMetrics     = field(default_factory=AnswerMetrics)\n    chunking:    ChunkingMetrics   = field(default_factory=ChunkingMetrics)\n    system:      SystemMetrics     = field(default_factory=SystemMetrics)\n    verdict:     str               = ""\n\n    def flat(self) -> dict:\n        d = {\n            "paper_id": self.paper_id,\n            "question": self.question[:60] + "...",\n            "strategy": self.strategy,\n            "target_size": self.target_size,\n            "top_k": self.top_k,\n            "prompt_style": self.prompt_style,\n            "verdict": self.verdict,\n        }\n        d.update(self.retrieval.to_dict())\n        d.update(self.answer.to_dict())\n        d.update(self.chunking.to_dict())\n        d.update(self.system.to_dict())\n        return d\n\n\n# ---------------------------------------------------------------------------\n# Retrieval metrics\n# ---------------------------------------------------------------------------\n\ndef recall_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:\n    top = retrieved_ids[:k]\n    return len(set(top) & relevant_ids) / max(len(relevant_ids), 1)\n\n\ndef precision_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:\n    top = retrieved_ids[:k]\n    return sum(1 for r in top if r in relevant_ids) / max(k, 1)\n\n\ndef compute_retrieval_metrics(retrieved_ids: List[str],\n                               relevant_ids: Set[str],\n                               ks: tuple = (2, 3, 5),\n                               context_relevance: float = 0.0) -> RetrievalMetrics:\n    m = RetrievalMetrics(context_relevance=context_relevance)\n    for k in ks:\n        m.recall[k]    = recall_at_k(retrieved_ids, relevant_ids, k)\n        m.precision[k] = precision_at_k(retrieved_ids, relevant_ids, k)\n    return m\n\n\n# ---------------------------------------------------------------------------\n# Answer quality metrics\n# ---------------------------------------------------------------------------\n\ndef compute_answer_metrics(structured_answer,\n                            hall_report,\n                            context_chunks: List[str],\n                            embedder) -> AnswerMetrics:\n    """\n    faithfulness   — avg cosine sim between answer sentences and context\n    completeness   — fraction of required sections filled\n    hallucination_rate — from hallucination detector\n    """\n    # Faithfulness via embedding cosine sim\n    ans_text = " ".join(structured_answer.to_dict().values())\n    ctx_text = " ".join(context_chunks)\n    a_vec = embedder.encode_one(ans_text)\n    c_vec = embedder.encode_one(ctx_text)\n    faithfulness = max(0.0, float(np.dot(a_vec, c_vec)))\n\n    return AnswerMetrics(\n        faithfulness=faithfulness,\n        completeness=structured_answer.completeness(),\n        hallucination_rate=hall_report.score,\n    )\n\n\n# ---------------------------------------------------------------------------\n# Chunking metrics\n# ---------------------------------------------------------------------------\n\ndef compute_chunking_metrics(chunks: List[object],\n                              source_text: str,\n                              embedder) -> ChunkingMetrics:\n    if not chunks:\n        return ChunkingMetrics()\n\n    texts = [c.text for c in chunks]\n    vecs = embedder.encode(texts)  # (N, dim)\n\n    # Coherence: avg cosine of adjacent pairs\n    adj_sims = [float(np.dot(vecs[i], vecs[i + 1])) for i in range(len(vecs) - 1)]\n    coherence = float(np.mean(adj_sims)) if adj_sims else 1.0\n\n    # Redundancy: avg cosine of sampled non-adjacent pairs\n    n = len(vecs)\n    non_adj = []\n    for i in range(n):\n        for j in range(i + 2, min(i + 6, n)):\n            non_adj.append(float(np.dot(vecs[i], vecs[j])))\n    redundancy = float(np.mean(non_adj)) if non_adj else 0.0\n\n    # Coverage: fraction of source vocabulary covered by chunks\n    src_words   = set(source_text.lower().split())\n    chunk_words = set()\n    for c in chunks:\n        chunk_words.update(c.text.lower().split())\n    coverage = len(chunk_words & src_words) / max(len(src_words), 1)\n\n    return ChunkingMetrics(\n        coherence=max(0.0, coherence),\n        redundancy=max(0.0, redundancy),\n        coverage=min(1.0, coverage),\n    )\n\n\n# ---------------------------------------------------------------------------\n# Evaluator class (convenience wrapper)\n# ---------------------------------------------------------------------------\n\nclass Evaluator:\n    """\n    Stateful evaluator that accumulates EvalRecords.\n\n    Usage\n    -----\n    ev = Evaluator(embedder)\n    record = ev.evaluate(paper_id, question, chunks, retrieved, answer, hall_report, ...)\n    df = ev.as_dataframe()\n    """\n\n    def __init__(self, embedder):\n        self.embedder = embedder\n        self.records: List[EvalRecord] = []\n\n    def evaluate(self,\n                 paper_id: str,\n                 question: str,\n                 strategy: str,\n                 target_size: int,\n                 top_k: int,\n                 prompt_style: str,\n                 all_chunks: List[object],\n                 retrieved: List[object],         # SearchResult objects\n                 structured_answer,\n                 hall_report,\n                 source_text: str,\n                 start_time: float,\n                 relevant_ids: Optional[Set[str]] = None) -> EvalRecord:\n\n        ctx_chunks = [r.chunk.text for r in retrieved]\n\n        # Retrieval metrics\n        retrieved_ids = [r.chunk.chunk_id for r in retrieved]\n        rel_ids = relevant_ids or {retrieved_ids[0]} if retrieved_ids else set()\n        ctx_rel = self._context_relevance(question, ctx_chunks)\n        ret_m = compute_retrieval_metrics(retrieved_ids, rel_ids,\n                                          ks=(2, 3, 5),\n                                          context_relevance=ctx_rel)\n\n        # Answer metrics\n        ans_m = compute_answer_metrics(structured_answer, hall_report,\n                                       ctx_chunks, self.embedder)\n\n        # Chunking metrics\n        ck_m = compute_chunking_metrics(all_chunks, source_text, self.embedder)\n\n        # System metrics\n        sys_m = SystemMetrics(\n            latency_ms=(time.time() - start_time) * 1000,\n            tokens_used=structured_answer.tokens_used,\n            chunks_retrieved=len(retrieved),\n        )\n\n        record = EvalRecord(\n            paper_id=paper_id,\n            question=question,\n            strategy=strategy,\n            target_size=target_size,\n            top_k=top_k,\n            prompt_style=prompt_style,\n            retrieval=ret_m,\n            answer=ans_m,\n            chunking=ck_m,\n            system=sys_m,\n            verdict=hall_report.verdict,\n        )\n        self.records.append(record)\n        return record\n\n    def _context_relevance(self, query: str, ctx_chunks: List[str]) -> float:\n        if not ctx_chunks:\n            return 0.0\n        q_vec = self.embedder.encode_one(query)\n        scores = []\n        for text in ctx_chunks:\n            c_vec = self.embedder.encode_one(text)\n            scores.append(float(np.dot(q_vec, c_vec)))\n        return float(np.mean(scores))\n\n    def as_dataframe(self):\n        import pandas as pd\n        return pd.DataFrame([r.flat() for r in self.records])\n\n    def summary(self) -> dict:\n        if not self.records:\n            return {}\n        import pandas as pd\n        df = self.as_dataframe()\n        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()\n        return df[num_cols].mean().round(4).to_dict()\n')
print('  wrote evaluation.py')
open('pipeline.py','w').write('"""\npipeline.py\n===========\nCentral RAG pipeline orchestrator.\n\nFlow\n----\n  PaperDoc → chunks → embeddings → FAISS → retrieve → LLM → answer\n                                                           ↓\n                                              HallucinationDetector → report\n\nDataset discipline\n------------------\n  load_train_test() → loads Attention + BERT for system development\n  load_validation() → loads RAG paper ONLY for final validation\n"""\n\nimport logging\nimport time\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional\n\nfrom dataset     import PaperDoc, load_paper, load_train_test, load_validation\nfrom chunking    import get_chunks, Chunk, STRATEGIES as CHUNK_STRATEGIES\nfrom retrieval   import RetrievalPipeline, SearchResult\nfrom llm         import LLMEngine, StructuredAnswer\nfrom hallucination import HallucinationDetector, HallucinationReport\n\nlogger = logging.getLogger(__name__)\n\n\n# ---------------------------------------------------------------------------\n# Config\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass PipelineConfig:\n    # Chunking\n    chunk_strategy:  str = "dynamic"\n    chunk_size:      int = 500        # tokens (target / max depending on strategy)\n    chunk_overlap:   int = 100        # used by overlapping strategy only\n\n    # Retrieval\n    top_k:           int = 5\n    use_reranker:    bool = False\n    embed_model:     str = "sentence-transformers/all-MiniLM-L6-v2"\n\n    # LLM\n    llm_backend:     str = "mock"     # "mock" | "tinyllama"\n    prompt_style:    str = "strict"   # "strict" | "open"\n    max_new_tokens:  int = 512\n\n    # Runtime\n    device:          str = "cpu"\n    data_dir:        str = "data"\n\n\n# ---------------------------------------------------------------------------\n# Pipeline response\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass PipelineResponse:\n    paper_id:   str\n    question:   str\n    answer:     StructuredAnswer\n    retrieved:  List[SearchResult]\n    hall_report: HallucinationReport\n    latency_ms: float\n    config:     dict = field(default_factory=dict)\n\n    @property\n    def context_text(self) -> str:\n        return " ".join(r.chunk.text for r in self.retrieved)\n\n    def summary(self) -> str:\n        lines = [\n            f"Paper    : {self.paper_id}",\n            f"Question : {self.question}",\n            f"Latency  : {self.latency_ms:.0f}ms",\n            f"Chunks   : {len(self.retrieved)}",\n            "",\n            str(self.answer),\n            "",\n            self.hall_report.summary(),\n        ]\n        return "\\n".join(lines)\n\n\n# ---------------------------------------------------------------------------\n# Core pipeline\n# ---------------------------------------------------------------------------\n\nclass RAGPipeline:\n    """\n    End-to-end RAG pipeline.\n\n    Usage\n    -----\n    pipeline = RAGPipeline(PipelineConfig(llm_backend="mock"))\n    pipeline.load_paper("attention")\n    pipeline.load_paper("bert")\n    response = pipeline.query("attention", "What is multi-head attention?")\n    """\n\n    def __init__(self, config: Optional[PipelineConfig] = None):\n        self.config = config or PipelineConfig()\n        self._docs:   Dict[str, PaperDoc] = {}\n        self._chunks: Dict[str, List[Chunk]] = {}\n        self._retrieval = RetrievalPipeline(\n            embed_model=self.config.embed_model,\n            use_reranker=self.config.use_reranker,\n            device=self.config.device,\n        )\n        self._llm = LLMEngine(backend=self.config.llm_backend)\n        self._detector = HallucinationDetector(\n            embedder=self._retrieval.embedder,\n            llm_engine=self._llm,\n        )\n        logger.info(f"RAGPipeline ready: {self.config}")\n\n    # ── Paper management ────────────────────────────────────────────────────\n\n    def load_paper(self, paper_id: str) -> PaperDoc:\n        """Download, extract, chunk, and index one paper."""\n        if paper_id in self._docs:\n            logger.info(f"[{paper_id}] already loaded — skipping.")\n            return self._docs[paper_id]\n\n        doc = load_paper(paper_id, self.config.data_dir)\n        self._docs[paper_id] = doc\n\n        chunks = self._make_chunks(doc)\n        self._chunks[paper_id] = chunks\n\n        logger.info(f"[{paper_id}] indexing {len(chunks)} chunks …")\n        self._retrieval.index_chunks(chunks, show_progress=True)\n        logger.info(f"[{paper_id}] indexed ✓")\n        return doc\n\n    def load_train_test(self) -> Dict[str, PaperDoc]:\n        docs = load_train_test(self.config.data_dir)\n        for pid, doc in docs.items():\n            if pid not in self._docs:\n                self._docs[pid] = doc\n                chunks = self._make_chunks(doc)\n                self._chunks[pid] = chunks\n                self._retrieval.index_chunks(chunks, show_progress=True)\n        return docs\n\n    def load_validation_paper(self) -> PaperDoc:\n        """Load RAG paper — call ONLY during validation phase."""\n        doc = load_validation(self.config.data_dir)\n        self._docs[doc.paper_id] = doc\n        chunks = self._make_chunks(doc)\n        self._chunks[doc.paper_id] = chunks\n        self._retrieval.index_chunks(chunks, show_progress=True)\n        return doc\n\n    # ── Chunking ─────────────────────────────────────────────────────────────\n\n    def _make_chunks(self, doc: PaperDoc,\n                     strategy: Optional[str] = None,\n                     size: Optional[int] = None) -> List[Chunk]:\n        strat = strategy or self.config.chunk_strategy\n        sz    = size    or self.config.chunk_size\n        kwargs = {}\n        if strat == "fixed":        kwargs = {"size": sz}\n        elif strat == "sentence":   kwargs = {"size": sz}\n        elif strat == "dynamic":    kwargs = {"lo": max(50, sz // 4), "hi": sz}\n        elif strat == "overlapping":kwargs = {"size": sz, "overlap": self.config.chunk_overlap}\n        elif strat == "heading":    kwargs = {"max_size": sz}\n\n        return get_chunks(doc.full_text, doc.paper_id, strategy=strat, **kwargs)\n\n    def rechunk(self, paper_id: str, strategy: str, size: int) -> List[Chunk]:\n        """Reindex a paper with a different chunking config (for experiments)."""\n        if paper_id not in self._docs:\n            raise ValueError(f"Paper {paper_id!r} not loaded.")\n        self._retrieval.reset_index()\n        # Re-index ALL loaded papers with new config\n        for pid, doc in self._docs.items():\n            chunks = self._make_chunks(doc, strategy=strategy, size=size)\n            self._chunks[pid] = chunks\n            self._retrieval.index_chunks(chunks, show_progress=False)\n        return self._chunks[paper_id]\n\n    # ── Query ────────────────────────────────────────────────────────────────\n\n    def query(self,\n              paper_id: str,\n              question: str,\n              top_k: Optional[int] = None,\n              prompt_style: Optional[str] = None) -> PipelineResponse:\n        """\n        Full RAG pipeline:\n          retrieve top-k chunks → LLM answer → hallucination detection\n        """\n        if paper_id not in self._docs:\n            raise ValueError(\n                f"Paper {paper_id!r} not loaded. Call load_paper(\'{paper_id}\') first."\n            )\n\n        t0 = time.time()\n        k     = top_k or self.config.top_k\n        style = prompt_style or self.config.prompt_style\n\n        # 1. Retrieve\n        retrieved = self._retrieval.retrieve(question, k=k, paper_id=paper_id)\n        context_chunks = [r.chunk.text for r in retrieved]\n        if not context_chunks:\n            context_chunks = ["No relevant context found in the paper."]\n\n        # 2. Generate\n        answer = self._llm.answer(\n            question, context_chunks,\n            style=style,\n            max_new_tokens=self.config.max_new_tokens,\n        )\n\n        # 3. Hallucination detection\n        ctx_text = " ".join(context_chunks)\n        report = self._detector.detect(answer.raw_text, ctx_text)\n\n        latency_ms = (time.time() - t0) * 1000\n\n        return PipelineResponse(\n            paper_id=paper_id,\n            question=question,\n            answer=answer,\n            retrieved=retrieved,\n            hall_report=report,\n            latency_ms=latency_ms,\n            config={\n                "strategy": self.config.chunk_strategy,\n                "size":     self.config.chunk_size,\n                "top_k":    k,\n                "style":    style,\n                "llm":      self.config.llm_backend,\n            },\n        )\n\n    def query_multi(self, question: str, paper_ids: List[str],\n                    k: int = 3) -> Dict[str, PipelineResponse]:\n        """Same question across multiple papers."""\n        return {pid: self.query(pid, question, top_k=k) for pid in paper_ids}\n\n    # ── Accessors ────────────────────────────────────────────────────────────\n\n    @property\n    def loaded_papers(self) -> List[str]:\n        return list(self._docs.keys())\n\n    def get_chunks(self, paper_id: str) -> List[Chunk]:\n        return self._chunks.get(paper_id, [])\n\n    def get_doc(self, paper_id: str) -> PaperDoc:\n        return self._docs[paper_id]\n\n\n# ---------------------------------------------------------------------------\n# Factory helpers\n# ---------------------------------------------------------------------------\n\ndef build_pipeline(backend: str = "mock",\n                   strategy: str = "dynamic",\n                   chunk_size: int = 500,\n                   top_k: int = 5,\n                   use_reranker: bool = False,\n                   prompt_style: str = "strict",\n                   data_dir: str = "data") -> RAGPipeline:\n    cfg = PipelineConfig(\n        llm_backend=backend,\n        chunk_strategy=strategy,\n        chunk_size=chunk_size,\n        top_k=top_k,\n        use_reranker=use_reranker,\n        prompt_style=prompt_style,\n        data_dir=data_dir,\n    )\n    return RAGPipeline(cfg)\n\n\nif __name__ == "__main__":\n    pipeline = build_pipeline(backend="mock", strategy="dynamic", chunk_size=500)\n    pipeline.load_paper("attention")\n    pipeline.load_paper("bert")\n\n    resp = pipeline.query("attention", "What is multi-head attention?")\n    print(resp.summary())\n')
print('  wrote pipeline.py')
open('experiments.py','w').write('"""\nexperiments.py\n==============\nSystematic ablation experiments over:\n  • chunk sizes   : 200, 500, 800 tokens\n  • top-k         : 2, 3, 5\n  • prompt styles : strict, open\n\nPhase 1 (Train/Test) — Attention + BERT papers only.\nPhase 2 (Validation) — RAG paper only (unseen during tuning).\n"""\n\nimport logging\nimport time\nfrom typing import List, Optional\n\nimport pandas as pd\n\nfrom pipeline  import RAGPipeline, PipelineConfig, build_pipeline\nfrom evaluation import Evaluator\nfrom chunking  import compare_strategies\n\nlogger = logging.getLogger(__name__)\n\n# ---------------------------------------------------------------------------\n# Benchmark queries\n# ---------------------------------------------------------------------------\n\nTRAIN_QUERIES = {\n    "attention": [\n        "What is the core innovation of the Transformer architecture?",\n        "How does scaled dot-product attention work?",\n        "What are the limitations of the Transformer model?",\n        "How does multi-head attention differ from single-head attention?",\n    ],\n    "bert": [\n        "What pre-training objectives does BERT use?",\n        "How does BERT handle fine-tuning for downstream tasks?",\n        "What is the difference between BERT-base and BERT-large?",\n        "What are the main limitations of BERT?",\n    ],\n}\n\nVALIDATION_QUERIES = {\n    "rag": [\n        "How does RAG combine retrieval with generation?",\n        "What retrieval mechanism does RAG use?",\n        "What are the main limitations of the RAG approach?",\n        "How does RAG differ from standard seq2seq models?",\n        "What benchmarks does RAG improve upon?",\n    ],\n}\n\n\n# ---------------------------------------------------------------------------\n# Experiment runner\n# ---------------------------------------------------------------------------\n\nclass ExperimentRunner:\n    """\n    Runs all ablation experiments and collects results.\n\n    Usage\n    -----\n    runner = ExperimentRunner(backend="mock", data_dir="data")\n    runner.run_train_test()          # Attention + BERT\n    runner.run_validation()          # RAG paper (call after train/test)\n    df = runner.results_df()\n    """\n\n    CHUNK_SIZES   = [200, 500, 800]\n    TOP_K_VALUES  = [2, 3, 5]\n    PROMPT_STYLES = ["strict", "open"]\n    STRATEGIES    = ["dynamic", "sentence", "overlapping"]  # key strategies\n\n    def __init__(self, backend: str = "mock", data_dir: str = "data",\n                 use_reranker: bool = False):\n        self.backend      = backend\n        self.data_dir     = data_dir\n        self.use_reranker = use_reranker\n        self._records: List[dict] = []\n\n    # ── Phase 1: Train/Test ──────────────────────────────────────────────────\n\n    def run_train_test(self, verbose: bool = True) -> pd.DataFrame:\n        """\n        Experiment 1 — chunk sizes (200/500/800) with best strategy.\n        Experiment 2 — top-k values (2/3/5).\n        Experiment 3 — prompt styles (strict/open).\n        """\n        print("=" * 60)\n        print("PHASE 1: Train/Test (Attention + BERT)")\n        print("=" * 60)\n\n        # Build base pipeline and load papers once\n        base = build_pipeline(backend=self.backend, data_dir=self.data_dir,\n                              use_reranker=self.use_reranker)\n        base.load_paper("attention")\n        base.load_paper("bert")\n        embedder = base._retrieval.embedder\n\n        # ── EXP 1: Chunk sizes ──────────────────────────────────────────────\n        print("\\n[EXP 1] Chunk size comparison: 200 / 500 / 800 tokens")\n        for size in self.CHUNK_SIZES:\n            self._run_batch(\n                paper_ids=["attention", "bert"],\n                strategy="dynamic",\n                chunk_size=size,\n                top_k=5,\n                prompt_style="strict",\n                exp_name="chunk_size",\n                verbose=verbose,\n            )\n\n        # ── EXP 2: Top-k ────────────────────────────────────────────────────\n        print("\\n[EXP 2] Top-k retrieval: k = 2, 3, 5")\n        for k in self.TOP_K_VALUES:\n            self._run_batch(\n                paper_ids=["attention", "bert"],\n                strategy="dynamic",\n                chunk_size=500,\n                top_k=k,\n                prompt_style="strict",\n                exp_name="top_k",\n                verbose=verbose,\n            )\n\n        # ── EXP 3: Prompt styles ─────────────────────────────────────────────\n        print("\\n[EXP 3] Prompt style: strict vs open")\n        for style in self.PROMPT_STYLES:\n            self._run_batch(\n                paper_ids=["attention", "bert"],\n                strategy="dynamic",\n                chunk_size=500,\n                top_k=5,\n                prompt_style=style,\n                exp_name="prompt_style",\n                verbose=verbose,\n            )\n\n        df = self.results_df()\n        print(f"\\nTrain/Test complete — {len(df)} records collected.")\n        return df\n\n    # ── Phase 2: Validation ──────────────────────────────────────────────────\n\n    def run_validation(self, verbose: bool = True) -> pd.DataFrame:\n        """\n        Evaluate best config (dynamic/500/k=5/strict) on unseen RAG paper.\n        """\n        print("\\n" + "=" * 60)\n        print("PHASE 2: Validation — RAG paper (UNSEEN)")\n        print("=" * 60)\n\n        self._run_batch(\n            paper_ids=["rag"],\n            strategy="dynamic",\n            chunk_size=500,\n            top_k=5,\n            prompt_style="strict",\n            exp_name="validation",\n            verbose=verbose,\n        )\n\n        val_df = self.results_df()\n        val_df = val_df[val_df["exp_name"] == "validation"]\n        print(f"\\nValidation complete — {len(val_df)} records.")\n        return val_df\n\n    # ── Internal runner ──────────────────────────────────────────────────────\n\n    def _run_batch(self, paper_ids: List[str], strategy: str,\n                   chunk_size: int, top_k: int, prompt_style: str,\n                   exp_name: str, verbose: bool = True) -> None:\n        """Build a fresh pipeline, load papers, run queries, collect metrics."""\n        cfg = PipelineConfig(\n            llm_backend=self.backend,\n            chunk_strategy=strategy,\n            chunk_size=chunk_size,\n            top_k=top_k,\n            use_reranker=self.use_reranker,\n            prompt_style=prompt_style,\n            data_dir=self.data_dir,\n        )\n        pipeline = RAGPipeline(cfg)\n        evaluator = Evaluator(pipeline._retrieval.embedder)\n\n        # Load papers\n        for pid in paper_ids:\n            pipeline.load_paper(pid)\n\n        # Run queries\n        query_map = VALIDATION_QUERIES if "rag" in paper_ids else TRAIN_QUERIES\n        for pid in paper_ids:\n            queries = query_map.get(pid, [])\n            for question in queries:\n                t0 = time.time()\n                try:\n                    response = pipeline.query(pid, question,\n                                              top_k=top_k,\n                                              prompt_style=prompt_style)\n                    chunks = pipeline.get_chunks(pid)\n                    record = evaluator.evaluate(\n                        paper_id=pid,\n                        question=question,\n                        strategy=strategy,\n                        target_size=chunk_size,\n                        top_k=top_k,\n                        prompt_style=prompt_style,\n                        all_chunks=chunks,\n                        retrieved=response.retrieved,\n                        structured_answer=response.answer,\n                        hall_report=response.hall_report,\n                        source_text=pipeline.get_doc(pid).full_text[:5000],\n                        start_time=t0,\n                    )\n                    row = record.flat()\n                    row["exp_name"] = exp_name\n                    row["latency_ms"] = response.latency_ms\n                    self._records.append(row)\n\n                    if verbose:\n                        print(\n                            f"  {pid:12s} sz={chunk_size:4d} k={top_k} "\n                            f"style={prompt_style:7s} → "\n                            f"hall={response.hall_report.score:.3f}  "\n                            f"complete={response.answer.completeness():.2f}  "\n                            f"{response.hall_report.verdict}"\n                        )\n                except Exception as e:\n                    logger.error(f"[{pid}] query failed: {e}")\n\n    # ── Results ──────────────────────────────────────────────────────────────\n\n    def results_df(self) -> pd.DataFrame:\n        if not self._records:\n            return pd.DataFrame()\n        return pd.DataFrame(self._records)\n\n    def print_summary(self) -> None:\n        df = self.results_df()\n        if df.empty:\n            print("No results yet.")\n            return\n\n        num_cols = [c for c in ["hallucination_rate", "completeness", "faithfulness",\n                                 "context_relevance", "latency_ms"]\n                    if c in df.columns]\n\n        print("\\n" + "=" * 60)\n        print("EXPERIMENT SUMMARY")\n        print("=" * 60)\n\n        for exp_name, grp in df.groupby("exp_name"):\n            print(f"\\n── {exp_name} ──")\n            vary_col = {"chunk_size": "target_size", "top_k": "top_k",\n                        "prompt_style": "prompt_style", "validation": "paper_id"}.get(\n                exp_name, "paper_id"\n            )\n            if vary_col in grp.columns:\n                sub = grp.groupby(vary_col)[num_cols].mean().round(3)\n                print(sub.to_string())\n\n    def chunking_comparison_table(self, text: str, paper_id: str) -> pd.DataFrame:\n        """Return chunking strategy stats as a DataFrame."""\n        rows = compare_strategies(text, paper_id, sizes=self.CHUNK_SIZES)\n        return pd.DataFrame([r for r in rows if "error" not in r])\n\n    def save_results(self, path: str = "experiment_results.csv") -> None:\n        df = self.results_df()\n        df.to_csv(path, index=False)\n        print(f"Results saved → {path}  ({len(df)} rows)")\n\n\n# ---------------------------------------------------------------------------\n# CLI\n# ---------------------------------------------------------------------------\n\nif __name__ == "__main__":\n    runner = ExperimentRunner(backend="mock", data_dir="data")\n    runner.run_train_test(verbose=True)\n    runner.run_validation(verbose=True)\n    runner.print_summary()\n    runner.save_results()\n')
print('  wrote experiments.py')
open('comparison.py','w').write('"""\ncomparison.py\n=============\nMulti-paper comparison engine.\n\nCompares papers on:  Problem · Method · Results · Limitations · When to use\n"""\n\nimport logging\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional\n\nlogger = logging.getLogger(__name__)\n\n# ---------------------------------------------------------------------------\n# Static knowledge base (always-available; for UI even without LLM)\n# ---------------------------------------------------------------------------\n\nSTATIC_PROFILES: Dict[str, Dict[str, str]] = {\n    "attention": {\n        "Problem":      "Sequence-to-sequence transduction (translation, summarisation) relying on recurrent / convolutional networks is slow to train and hard to parallelise.",\n        "Method":       "Pure attention-based encoder-decoder (Transformer). Multi-head self-attention + position-wise FFN + positional encodings replace recurrence entirely.",\n        "Results":      "BLEU 28.4 on WMT-14 EN→DE (new SOTA); BLEU 41.0 on EN→FR. ~8× faster training than best RNN ensemble. Attention patterns are interpretable.",\n        "Limitations":  "Quadratic O(n²) attention complexity limits long-sequence scalability. Fixed maximum context window. Requires large parallel corpora.",\n        "When to use":  "Machine translation, text generation, seq2seq tasks where training speed matters and sequences are ≤ 512 tokens.",\n    },\n    "bert": {\n        "Problem":      "Pre-training language representations uni-directionally (left-to-right or shallow concatenation) limits the power of learned representations for NLU tasks.",\n        "Method":       "Bidirectional Transformer pre-trained with Masked Language Modeling (MLM) and Next Sentence Prediction (NSP). Fine-tuned end-to-end with a task head.",\n        "Results":      "Absolute improvements on 11 NLP tasks: GLUE +7.7%, SQuAD 1.1 F1 93.2%, SQuAD 2.0 F1 83.1%, MultiNLI +4.7%.",\n        "Limitations":  "Cannot generate text natively (encoder-only). NSP task shown to be noisy (cf. RoBERTa). Heavy pre-training cost. [MASK] not seen at fine-tune time.",\n        "When to use":  "Text classification, NER, extractive QA, sentence-pair tasks, feature extraction — any NLU task with labelled fine-tune data.",\n    },\n    "rag": {\n        "Problem":      "Parametric knowledge in LLM weights is static, opaque, and hard to update. Knowledge-intensive tasks require up-to-date, verifiable facts.",\n        "Method":       "Non-parametric retrieval (Dense Passage Retrieval) + parametric generation (BART/seq2seq). Retrieves top-k passages; generator conditions on them.",\n        "Results":      "Outperforms specialised SOTA on Natural Questions, WebQ, TriviaQA (open-domain); SOTA on MS-MARCO NLG; strong on Jeopardy question generation.",\n        "Limitations":  "Retrieval quality is a bottleneck. Joint training is complex. Inference speed depends on retrieval corpus size. No real-time corpus update during generation.",\n        "When to use":  "Open-domain QA, fact verification, knowledge-intensive NLG — any task needing factual grounding beyond what fits in LLM weights.",\n    },\n}\n\nDIMENSIONS = ["Problem", "Method", "Results", "Limitations", "When to use"]\n\n\n@dataclass\nclass ComparisonTable:\n    paper_ids: List[str]\n    data: Dict[str, Dict[str, str]]   # {dimension: {paper_id: text}}\n    recommendation: str = ""\n\n    def to_markdown(self) -> str:\n        titles = {pid: STATIC_PROFILES.get(pid, {}).get("title", pid)\n                  for pid in self.paper_ids}\n        header = "| Dimension | " + " | ".join(self.paper_ids) + " |"\n        sep    = "|---|" + "---|" * len(self.paper_ids)\n        rows   = [header, sep]\n        for dim in DIMENSIONS:\n            cells = " | ".join(\n                self.data.get(dim, {}).get(pid, "N/A")[:120]\n                for pid in self.paper_ids\n            )\n            rows.append(f"| **{dim}** | {cells} |")\n        if self.recommendation:\n            rows.append(f"\\n**Recommendation:** {self.recommendation}")\n        return "\\n".join(rows)\n\n    def as_dict_list(self) -> List[dict]:\n        rows = []\n        for dim in DIMENSIONS:\n            row = {"Dimension": dim}\n            for pid in self.paper_ids:\n                row[pid] = self.data.get(dim, {}).get(pid, "N/A")\n            rows.append(row)\n        return rows\n\n\nclass ComparisonEngine:\n    """\n    Build a ComparisonTable for a set of papers.\n    Uses static profiles when available; can augment with LLM-retrieved answers.\n    """\n\n    def __init__(self, pipeline=None):\n        self._pipeline = pipeline\n\n    def compare(self, paper_ids: List[str]) -> ComparisonTable:\n        data: Dict[str, Dict[str, str]] = {dim: {} for dim in DIMENSIONS}\n\n        for pid in paper_ids:\n            profile = STATIC_PROFILES.get(pid, {})\n            for dim in DIMENSIONS:\n                data[dim][pid] = profile.get(dim, "Not available in static profile.")\n\n        recommendation = self._generate_recommendation(paper_ids)\n        return ComparisonTable(\n            paper_ids=paper_ids,\n            data=data,\n            recommendation=recommendation,\n        )\n\n    def _generate_recommendation(self, paper_ids: List[str]) -> str:\n        mapping = {\n            frozenset(["attention", "bert"]): (\n                "Use **Transformer** for generation/translation tasks. "\n                "Use **BERT** for classification/NLU. Both are complementary."\n            ),\n            frozenset(["attention", "rag"]): (\n                "Use **Transformer** for closed-context generation. "\n                "Use **RAG** when external, up-to-date knowledge is required."\n            ),\n            frozenset(["bert", "rag"]): (\n                "Use **BERT** for discriminative NLU with labelled data. "\n                "Use **RAG** for open-domain QA without labelled answers."\n            ),\n            frozenset(["attention", "bert", "rag"]): (\n                "All three form a natural progression: Transformer → BERT → RAG. "\n                "Choose based on task: generation → Transformer; "\n                "classification → BERT; knowledge-intensive QA → RAG."\n            ),\n        }\n        return mapping.get(frozenset(paper_ids), "See individual profiles for task-specific guidance.")\n\n\nif __name__ == "__main__":\n    engine = ComparisonEngine()\n    table  = engine.compare(["attention", "bert", "rag"])\n    print(table.to_markdown())\n')
print('  wrote comparison.py')
open('question_generator.py','w').write('"""\nquestion_generator.py\n=====================\nGenerates 3–5 critical research questions per paper.\n\nCategories\n----------\n  assumption  — challenges a key assumption the paper makes\n  weakness    — identifies a known or suspected weakness\n  improvement — proposes a concrete improvement or extension\n"""\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional\n\n# ---------------------------------------------------------------------------\n# Data models\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass ResearchQuestion:\n    question:   str\n    category:   str      # assumption | weakness | improvement\n    rationale:  str = ""\n    difficulty: str = "medium"   # easy | medium | hard\n\n\n@dataclass\nclass QuestionSet:\n    paper_id: str\n    title:    str\n    questions: List[ResearchQuestion] = field(default_factory=list)\n\n    def to_markdown(self) -> str:\n        lines = [f"## Critical Research Questions: {self.title}", ""]\n        icons = {"assumption": "🔍", "weakness": "⚠️", "improvement": "💡"}\n        for i, q in enumerate(self.questions, 1):\n            icon = icons.get(q.category, "📌")\n            lines.append(f"{i}. {icon} **[{q.category.upper()} | {q.difficulty}]**")\n            lines.append(f"   **Q:** {q.question}")\n            if q.rationale:\n                lines.append(f"   *Why this matters:* {q.rationale}")\n            lines.append("")\n        return "\\n".join(lines)\n\n    def to_list(self) -> List[dict]:\n        return [\n            {\n                "question":  q.question,\n                "category":  q.category,\n                "rationale": q.rationale,\n                "difficulty": q.difficulty,\n            }\n            for q in self.questions\n        ]\n\n\n# ---------------------------------------------------------------------------\n# Static question bank\n# ---------------------------------------------------------------------------\n\n_BANK: dict = {\n    "attention": [\n        ResearchQuestion(\n            "Does the Transformer\'s quadratic O(n²) attention complexity represent a fundamental barrier for long-document processing, and can it be resolved without sacrificing expressivity?",\n            category="weakness",\n            rationale="Memory and compute scale quadratically with sequence length — a hard architectural limit for documents longer than ~512 tokens.",\n            difficulty="hard",\n        ),\n        ResearchQuestion(\n            "Are sinusoidal positional encodings truly sufficient, or do learned / relative position representations (RoPE, ALiBi) provide a strict improvement?",\n            category="assumption",\n            rationale="The paper assumes fixed sinusoidal encodings are adequate; subsequent work (T5, LLaMA) moved to learned or relative encodings.",\n            difficulty="medium",\n        ),\n        ResearchQuestion(\n            "Is multi-head attention strictly necessary, or does a single high-dimensional head with appropriate regularisation achieve equivalent performance?",\n            category="assumption",\n            rationale="The ablation in the paper is limited; the benefit of multiple heads vs. one large head is not fully theoretically justified.",\n            difficulty="medium",\n        ),\n        ResearchQuestion(\n            "Could sparse attention (Longformer, BigBird) or linear attention (Performer) match full attention quality while reducing complexity to O(n)?",\n            category="improvement",\n            rationale="Sparse attention is the direct next step toward scaling Transformers to long contexts.",\n            difficulty="hard",\n        ),\n        ResearchQuestion(\n            "How well does the Transformer generalise to truly low-resource language pairs where large parallel corpora are unavailable?",\n            category="weakness",\n            rationale="All results in the paper use large WMT datasets; data efficiency in low-resource regimes is untested.",\n            difficulty="easy",\n        ),\n    ],\n    "bert": [\n        ResearchQuestion(\n            "Does the Next Sentence Prediction (NSP) objective actually help, or does it introduce noise that hurts downstream performance?",\n            category="weakness",\n            rationale="RoBERTa later showed that removing NSP improves results — a direct challenge to BERT\'s pre-training design.",\n            difficulty="medium",\n        ),\n        ResearchQuestion(\n            "Does the [MASK] token at pre-training time create a harmful distributional shift, since [MASK] never appears during fine-tuning?",\n            category="assumption",\n            rationale="BERT uses [MASK] for 15% of tokens at pre-train time but never at fine-tune time — this train/test mismatch is a known flaw addressed by ELECTRA.",\n            difficulty="medium",\n        ),\n        ResearchQuestion(\n            "How can BERT-style models effectively handle inputs longer than 512 tokens without losing global cross-document context?",\n            category="improvement",\n            rationale="The 512-token hard limit is an architectural bottleneck for document-level NLP tasks.",\n            difficulty="hard",\n        ),\n        ResearchQuestion(\n            "Can BERT\'s pre-training be made significantly more sample-efficient to reduce the enormous GPU compute required?",\n            category="improvement",\n            rationale="BERT requires days of TPU/GPU training; reducing this cost is critical for democratising NLP research.",\n            difficulty="hard",\n        ),\n        ResearchQuestion(\n            "Are BERT\'s learned representations truly capturing deep semantics, or are they sophisticated surface-level pattern matching?",\n            category="assumption",\n            rationale="Probing studies show BERT captures syntax but deeper semantic understanding is debated.",\n            difficulty="easy",\n        ),\n    ],\n    "rag": [\n        ResearchQuestion(\n            "How does RAG behave when the dense retriever consistently returns irrelevant or adversarially poisoned passages?",\n            category="weakness",\n            rationale="RAG\'s output quality is entirely dependent on retrieval quality — a single point of failure that is not addressed in the paper.",\n            difficulty="medium",\n        ),\n        ResearchQuestion(\n            "Does joint training of the retriever and generator in RAG actually improve both components, or does one dominate and bottleneck the other?",\n            category="assumption",\n            rationale="The paper assumes joint training is beneficial, but ablating frozen-retriever vs. jointly-trained is only partially explored.",\n            difficulty="hard",\n        ),\n        ResearchQuestion(\n            "How can RAG be extended to multilingual or cross-lingual settings where the retrieval corpus and generation language differ?",\n            category="improvement",\n            rationale="The original RAG paper focuses exclusively on English; multilingual extension is an important open problem.",\n            difficulty="hard",\n        ),\n        ResearchQuestion(\n            "Can RAG models detect and resolve contradictions between retrieved passages without explicit contradiction-resolution training?",\n            category="weakness",\n            rationale="When multiple retrieved passages contain conflicting facts, the generator may produce inconsistent or averaged-out answers.",\n            difficulty="medium",\n        ),\n        ResearchQuestion(\n            "What is the optimal ratio of parametric knowledge (model weights) to non-parametric knowledge (retrieved passages) for maximum factual accuracy?",\n            category="improvement",\n            rationale="Understanding how much the model should \'trust\' its own weights vs. retrieved text is fundamental to improving RAG reliability.",\n            difficulty="hard",\n        ),\n    ],\n}\n\n\n# ---------------------------------------------------------------------------\n# Generator\n# ---------------------------------------------------------------------------\n\nclass QuestionGenerator:\n    """\n    Generates critical research questions for a paper.\n\n    Usage\n    -----\n    gen = QuestionGenerator()\n    qset = gen.generate("attention", n=5)\n    print(qset.to_markdown())\n    """\n\n    def __init__(self, pipeline=None):\n        self._pipeline = pipeline   # optional — reserved for LLM-based generation\n\n    def generate(self, paper_id: str, n: int = 5,\n                 categories: Optional[List[str]] = None) -> QuestionSet:\n        """\n        Return up to n critical questions for the given paper.\n\n        Parameters\n        ----------\n        paper_id   : "attention" | "bert" | "rag"\n        n          : number of questions (3–5 recommended)\n        categories : filter to specific categories (None = all)\n        """\n        from dataset import PAPERS\n        title = PAPERS.get(paper_id, {}).get("title", paper_id)\n\n        questions = _BANK.get(paper_id, [])\n        if categories:\n            questions = [q for q in questions if q.category in categories]\n        questions = questions[:n]\n\n        return QuestionSet(paper_id=paper_id, title=title, questions=questions)\n\n    def generate_for_all(self, paper_ids: Optional[List[str]] = None,\n                          n: int = 5) -> List[QuestionSet]:\n        pids = paper_ids or list(_BANK.keys())\n        return [self.generate(pid, n=n) for pid in pids]\n\n\nif __name__ == "__main__":\n    gen = QuestionGenerator()\n    for pid in ["attention", "bert", "rag"]:\n        qset = gen.generate(pid, n=5)\n        print(qset.to_markdown())\n        print("─" * 60)\n')
print('  wrote question_generator.py')
open('app.py','w').write('"""\napp.py\n======\nStreamlit UI for the AI Research Paper Explainer.\n\nTabs\n----\n  1. Query & Answer     — ask a question, get structured 5-section answer\n  2. Hallucination      — 3-layer hallucination report with gauges\n  3. Retrieved Chunks   — inspect what the model actually saw\n  4. Paper Comparison   — side-by-side across dimensions\n  5. Question Generator — critical research questions\n  6. Experiments        — ablation results dashboard\n  7. Design             — system design justification\n\nRun\n---\n  streamlit run app.py\n"""\n\nimport sys\nimport os\nimport time\nimport logging\n\nimport streamlit as st\nimport warnings\nwarnings.filterwarnings("ignore", message=".*use_container_width.*")\nwarnings.filterwarnings("ignore", message=".*torch_dtype.*")\nwarnings.filterwarnings("ignore", message=".*__path__.*")\nimport plotly.graph_objects as go\nimport plotly.express as px\nimport pandas as pd\n\n# ── Page config (must be first Streamlit call) ──────────────────────────────\nst.set_page_config(\n    page_title="AI Research Paper Explainer",\n    page_icon="🔬",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\n# ── Logging ──────────────────────────────────────────────────────────────────\nlogging.basicConfig(level=logging.WARNING)\n\n# ── CSS ──────────────────────────────────────────────────────────────────────\nst.markdown("""\n<style>\n@import url(\'https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600;700&display=swap\');\n\nhtml, body, [data-testid="stAppViewContainer"] {\n    background: #0f0f13 !important;\n    color: #e8e8f0 !important;\n    font-family: \'IBM Plex Sans\', sans-serif;\n}\n[data-testid="stSidebar"] {\n    background: #16161e !important;\n    border-right: 1px solid #2a2a3a;\n}\n.stButton > button {\n    background: #4f6ef7 !important;\n    color: #fff !important;\n    border: none !important;\n    border-radius: 3px !important;\n    font-family: \'IBM Plex Mono\', monospace !important;\n    font-weight: 600 !important;\n    letter-spacing: 0.04em;\n}\n.card {\n    background: #1a1a24;\n    border: 1px solid #2a2a3a;\n    border-left: 3px solid #4f6ef7;\n    padding: 1rem 1.2rem;\n    margin: 0.5rem 0;\n    border-radius: 3px;\n}\n.card-green  { border-left-color: #3ecf8e !important; }\n.card-yellow { border-left-color: #f5c542 !important; }\n.card-red    { border-left-color: #e05454 !important; }\n.card-purple { border-left-color: #a56eff !important; }\n.section-label {\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 0.68rem;\n    letter-spacing: 0.18em;\n    color: #7a8aff;\n    text-transform: uppercase;\n    margin-bottom: 0.35rem;\n}\n.chunk-box {\n    background: #111118;\n    border: 1px solid #252535;\n    border-left: 3px solid #a56eff;\n    padding: 0.7rem 0.9rem;\n    margin: 0.3rem 0;\n    font-family: \'IBM Plex Mono\', monospace;\n    font-size: 0.8rem;\n    line-height: 1.65;\n    border-radius: 2px;\n}\nh1, h2, h3 { font-family: \'IBM Plex Sans\', sans-serif !important; font-weight: 700 !important; }\n.verdict-GROUNDED     { color: #3ecf8e; font-weight: 700; }\n.verdict-PARTIAL      { color: #f5c542; font-weight: 700; }\n.verdict-HALLUCINATED { color: #e05454; font-weight: 700; }\n</style>\n""", unsafe_allow_html=True)\n\n\n# ---------------------------------------------------------------------------\n# Session state + pipeline init\n# ---------------------------------------------------------------------------\n\n@st.cache_resource(show_spinner="Initialising RAG pipeline…")\ndef _init_pipeline(backend: str, strategy: str, size: int, k: int, rerank: bool):\n    from pipeline import build_pipeline\n    return build_pipeline(\n        backend=backend, strategy=strategy, chunk_size=size,\n        top_k=k, use_reranker=rerank, prompt_style="strict",\n    )\n\n\ndef _get_pipeline():\n    cfg = st.session_state.get("cfg", {})\n    return _init_pipeline(\n        backend  = cfg.get("backend", "mock"),\n        strategy = cfg.get("strategy", "dynamic"),\n        size     = cfg.get("size", 500),\n        k        = cfg.get("k", 5),\n        rerank   = cfg.get("rerank", False),\n    )\n\n\ndef _ensure_loaded(pipeline, paper_id: str):\n    if paper_id not in pipeline.loaded_papers:\n        with st.spinner(f"Loading {paper_id} paper…"):\n            pipeline.load_paper(paper_id)\n\n\n# ---------------------------------------------------------------------------\n# Render helpers\n# ---------------------------------------------------------------------------\n\ndef _render_answer(answer, key_prefix: str = ""):\n    icons = {\n        "Core Idea":         "💡",\n        "Methodology":       "⚙️",\n        "Key Results":       "📊",\n        "Limitations":       "⚠️",\n        "ELI12 Explanation": "🎓",\n    }\n    for section, text in answer.to_dict().items():\n        icon = icons.get(section, "📌")\n        st.markdown(f"""\n        <div class="card">\n            <div class="section-label">{icon} {section}</div>\n            <div style="color:#dde;line-height:1.7;">{text}</div>\n        </div>""", unsafe_allow_html=True)\n\n\ndef _render_hallucination(report):\n    color_map = {"GROUNDED": "green", "PARTIAL": "yellow", "HALLUCINATED": "red"}\n    card_cls = f"card card-{color_map.get(report.verdict, \'purple\')}"\n\n    col1, col2, col3, col4 = st.columns(4)\n    with col1:\n        st.markdown(f"""<div class="{card_cls}">\n            <div class="section-label">Verdict</div>\n            <div class="verdict-{report.verdict}" style="font-size:1.3rem;">{report.verdict}</div>\n        </div>""", unsafe_allow_html=True)\n    with col2:\n        pct = int(report.score * 100)\n        st.markdown(f"""<div class="card">\n            <div class="section-label">Hall. Score</div>\n            <div style="font-size:1.3rem;font-weight:700;">{pct}%</div>\n        </div>""", unsafe_allow_html=True)\n    with col3:\n        pct2 = int(report.embed_similarity * 100)\n        st.markdown(f"""<div class="card">\n            <div class="section-label">Embed Sim</div>\n            <div style="font-size:1.3rem;font-weight:700;">{pct2}%</div>\n        </div>""", unsafe_allow_html=True)\n    with col4:\n        pct3 = int(report.claim_support_rate * 100)\n        st.markdown(f"""<div class="card">\n            <div class="section-label">Claim Support</div>\n            <div style="font-size:1.3rem;font-weight:700;">{pct3}%</div>\n        </div>""", unsafe_allow_html=True)\n\n    if report.unsupported_claims:\n        st.markdown("**⚠️ Unsupported claims:**")\n        for c in report.unsupported_claims[:3]:\n            st.markdown(f"- *{c[:120]}*")\n\n    # Gauge chart\n    fig = go.Figure(go.Indicator(\n        mode="gauge+number",\n        value=report.grounding_score * 100,\n        title={"text": "Grounding Score (%)", "font": {"color": "#aac", "size": 13}},\n        number={"font": {"color": "#eef"}},\n        gauge={\n            "axis": {"range": [0, 100], "tickcolor": "#444"},\n            "bar":  {"color": "#4f6ef7"},\n            "bgcolor": "#1a1a24",\n            "steps": [\n                {"range": [0,  40], "color": "#2a1a1a"},\n                {"range": [40, 65], "color": "#2a2a14"},\n                {"range": [65, 100], "color": "#162620"},\n            ],\n            "threshold": {"line": {"color": "#3ecf8e", "width": 3},\n                          "thickness": 0.75, "value": 65},\n        },\n    ))\n    fig.update_layout(paper_bgcolor="#0f0f13", font_color="#aab",\n                      height=220, margin=dict(t=40, b=0, l=20, r=20))\n    st.plotly_chart(fig, use_container_width=True)\n\n\ndef _render_chunks(retrieved):\n    st.markdown("### Retrieved Chunks")\n    for i, r in enumerate(retrieved):\n        score = r.rerank_score if r.rerank_score is not None else r.bi_score\n        with st.expander(\n            f"Chunk {i+1}  |  source={r.chunk.paper_id}  |  "\n            f"score={score:.4f}  |  tokens={r.chunk.token_count}  |  "\n            f"strategy={r.chunk.strategy}",\n            expanded=(i == 0),\n        ):\n            st.markdown(f\'<div class="chunk-box">{r.chunk.text}</div>\',\n                        unsafe_allow_html=True)\n            if r.chunk.section:\n                st.caption(f"📌 Section: {r.chunk.section}")\n\n\n# ---------------------------------------------------------------------------\n# Sidebar\n# ---------------------------------------------------------------------------\n\ndef sidebar():\n    with st.sidebar:\n        st.markdown("## ⚙️ Configuration")\n\n        PAPER_LABELS = {\n            "🧠 Attention Is All You Need": "attention",\n            "📚 BERT": "bert",\n            "🔍 RAG (Validation Only)": "rag",\n        }\n        paper_label = st.selectbox("Select Paper", list(PAPER_LABELS.keys()))\n        paper_id    = PAPER_LABELS[paper_label]\n\n        if paper_id == "rag":\n            st.warning("RAG is the **unseen validation** paper.")\n\n        st.divider()\n        backend  = st.radio("LLM Backend", ["mock", "tinyllama"],\n                            help="\'mock\' is instant; \'tinyllama\' needs GPU RAM")\n        strategy = st.selectbox("Chunking Strategy",\n                                ["dynamic", "sentence", "fixed", "overlapping", "heading"])\n        size     = st.select_slider("Chunk Size (tokens)",\n                                    options=[200, 300, 500, 800], value=500)\n        k        = st.slider("Top-k Chunks", 1, 10, 5)\n        style    = st.radio("Prompt Style", ["strict", "open"])\n        rerank   = st.checkbox("Use Reranker", value=False)\n\n        st.session_state["cfg"] = dict(\n            backend=backend, strategy=strategy, size=size,\n            k=k, style=style, rerank=rerank\n        )\n        st.divider()\n        st.markdown("### Dataset Split")\n        st.success("**Train/Test:** Attention · BERT")\n        st.warning("**Validation:** RAG paper only")\n        st.divider()\n        st.caption("Embedding: all-MiniLM-L6-v2")\n        st.caption("Vector DB: FAISS IndexFlatIP")\n        st.caption("LLM: TinyLlama-1.1B-Chat")\n\n    return paper_id, style\n\n\n# ---------------------------------------------------------------------------\n# Main\n# ---------------------------------------------------------------------------\n\ndef main():\n    st.markdown("""\n    <div style="padding:1.2rem 0 0.5rem 0; border-bottom:1px solid #2a2a3a; margin-bottom:1.2rem;">\n        <span style="font-family:\'IBM Plex Mono\',monospace;font-size:0.65rem;\n                     color:#4f6ef7;letter-spacing:0.22em;">RESEARCH INTELLIGENCE SYSTEM</span>\n        <h1 style="margin:0.2rem 0 0 0;font-size:1.9rem;">AI Research Paper Explainer</h1>\n        <p style="color:#6678aa;margin:0.2rem 0 0 0;font-size:0.88rem;">\n            Advanced RAG · FAISS · Hallucination Detection · Experiments Dashboard\n        </p>\n    </div>""", unsafe_allow_html=True)\n\n    paper_id, prompt_style = sidebar()\n\n    tabs = st.tabs([\n        "🔎 Query & Answer",\n        "🛡 Hallucination",\n        "📎 Chunks",\n        "📊 Comparison",\n        "❓ Questions",\n        "🔬 Experiments",\n        "📖 Design",\n    ])\n\n    # ── Tab 1: Query ─────────────────────────────────────────────────────────\n    with tabs[0]:\n        st.markdown(f"### Query: {paper_id.upper()}")\n        suggestions = {\n            "attention": [\n                "What is the core innovation of the Transformer?",\n                "How does multi-head attention work?",\n                "What are the limitations of the Transformer?",\n            ],\n            "bert": [\n                "What pre-training tasks does BERT use?",\n                "How does BERT fine-tune for downstream tasks?",\n                "What is masked language modeling?",\n            ],\n            "rag": [\n                "How does RAG combine retrieval with generation?",\n                "What is the retrieval mechanism in RAG?",\n                "What are the limitations of RAG?",\n            ],\n        }\n\n        st.markdown("**Quick questions:**")\n        q_cols = st.columns(3)\n        chosen_q = ""\n        for i, sq in enumerate(suggestions.get(paper_id, [])):\n            if q_cols[i].button(sq[:42] + "…", key=f"sq_{i}"):\n                chosen_q = sq\n\n        query = st.text_input("Or type your own question:",\n                               value=chosen_q, key="query_input")\n\n        if st.button("🔍 Ask", use_container_width=True) and query:\n            pipeline = _get_pipeline()\n            _ensure_loaded(pipeline, paper_id)\n\n            with st.spinner("Retrieving and generating…"):\n                try:\n                    resp = pipeline.query(paper_id, query,\n                                          top_k=st.session_state["cfg"].get("k", 5),\n                                          prompt_style=prompt_style)\n                    st.session_state["last_response"] = resp\n                except Exception as e:\n                    st.error(f"Error: {e}")\n                    return\n\n        resp = st.session_state.get("last_response")\n        if resp:\n            # Metrics bar\n            c1, c2, c3, c4 = st.columns(4)\n            c1.metric("⚡ Latency", f"{resp.latency_ms:.0f}ms")\n            c2.metric("🪙 Tokens", resp.answer.tokens_used)\n            c3.metric("📎 Chunks", len(resp.retrieved))\n            c4.metric("✅ Complete", f"{resp.answer.completeness():.0%}")\n\n            st.markdown("---")\n            col_a, col_b = st.columns([3, 2])\n            with col_a:\n                st.markdown("### Structured Answer")\n                _render_answer(resp.answer)\n            with col_b:\n                _render_chunks(resp.retrieved)\n\n    # ── Tab 2: Hallucination ─────────────────────────────────────────────────\n    with tabs[1]:\n        resp = st.session_state.get("last_response")\n        if not resp:\n            st.info("Ask a question in the Query tab first.")\n        else:\n            st.markdown("### Hallucination Analysis")\n            _render_hallucination(resp.hall_report)\n            with st.expander("Full report dict"):\n                st.json(resp.hall_report.to_dict())\n\n    # ── Tab 3: Chunks ────────────────────────────────────────────────────────\n    with tabs[2]:\n        resp = st.session_state.get("last_response")\n        if not resp:\n            st.info("Ask a question first.")\n        else:\n            _render_chunks(resp.retrieved)\n\n    # ── Tab 4: Comparison ────────────────────────────────────────────────────\n    with tabs[3]:\n        from comparison import ComparisonEngine\n        st.markdown("### Multi-Paper Comparison")\n\n        compare_ids = st.multiselect(\n            "Compare papers:",\n            ["attention", "bert", "rag"],\n            default=["attention", "bert"],\n        )\n        if st.button("🔄 Generate Comparison") and compare_ids:\n            engine = ComparisonEngine()\n            table  = engine.compare(compare_ids)\n            st.markdown(table.to_markdown())\n            df = pd.DataFrame(table.as_dict_list())\n            st.dataframe(df, use_container_width=True)\n\n        # Cross-paper query\n        st.markdown("---")\n        st.markdown("#### Same question, all papers")\n        shared_q = st.text_input("Question:", "What are the main limitations?",\n                                  key="shared_q")\n        if st.button("Compare across papers") and shared_q:\n            pipeline = _get_pipeline()\n            cols = st.columns(len(compare_ids))\n            for i, pid in enumerate(compare_ids):\n                _ensure_loaded(pipeline, pid)\n                with cols[i]:\n                    st.markdown(f"**{pid.upper()}**")\n                    try:\n                        r = pipeline.query(pid, shared_q, top_k=3)\n                        lim = r.answer.limitations\n                        st.markdown(f\'<div class="card">{lim[:280]}</div>\',\n                                    unsafe_allow_html=True)\n                        verdict_class = f"verdict-{r.hall_report.verdict}"\n                        st.markdown(\n                            f\'<span class="{verdict_class}">● {r.hall_report.verdict}</span>\',\n                            unsafe_allow_html=True,\n                        )\n                    except Exception as e:\n                        st.error(str(e))\n\n    # ── Tab 5: Question Generator ─────────────────────────────────────────────\n    with tabs[4]:\n        from question_generator import QuestionGenerator\n        st.markdown("### Critical Research Question Generator")\n\n        q_paper = st.selectbox("Paper:", ["attention", "bert", "rag"],\n                                key="q_paper_select")\n        n_q = st.slider("Number of questions:", 3, 5, 5, key="n_q_slider")\n\n        if st.button("Generate Questions"):\n            gen  = QuestionGenerator()\n            qset = gen.generate(q_paper, n=n_q)\n            icons = {"assumption": "🔍", "weakness": "⚠️", "improvement": "💡"}\n            diff_colors = {"easy": "#3ecf8e", "medium": "#f5c542", "hard": "#e05454"}\n            for q in qset.questions:\n                icon = icons.get(q.category, "📌")\n                dc   = diff_colors.get(q.difficulty, "#aaa")\n                st.markdown(f"""\n                <div class="card card-purple">\n                    <div style="display:flex;justify-content:space-between;">\n                        <span class="section-label">{icon} {q.category}</span>\n                        <span style="font-family:IBM Plex Mono;font-size:0.68rem;color:{dc};">\n                            ◆ {q.difficulty}\n                        </span>\n                    </div>\n                    <div style="font-size:0.97rem;font-weight:600;margin:0.4rem 0;">\n                        {q.question}\n                    </div>\n                    <div style="color:#8899bb;font-size:0.83rem;">→ {q.rationale}</div>\n                </div>""", unsafe_allow_html=True)\n\n    # ── Tab 6: Experiments ───────────────────────────────────────────────────\n    with tabs[5]:\n        st.markdown("### Experiments Dashboard")\n        st.markdown("""\n        Run ablation experiments across:\n        - **Chunk sizes**: 200 / 500 / 800 tokens\n        - **Top-k**: 2 / 3 / 5\n        - **Prompt styles**: strict / open\n        """)\n\n        col_run, col_val = st.columns(2)\n        with col_run:\n            if st.button("▶ Run Train/Test Experiments (mock, fast)"):\n                from experiments import ExperimentRunner\n                runner = ExperimentRunner(backend="mock")\n                with st.spinner("Running experiments…"):\n                    df = runner.run_train_test(verbose=False)\n                st.session_state["exp_df"] = df\n                st.success(f"Done — {len(df)} records")\n\n        with col_val:\n            if st.button("🎯 Run Validation (RAG paper)"):\n                from experiments import ExperimentRunner\n                runner = ExperimentRunner(backend="mock")\n                with st.spinner("Loading RAG paper and validating…"):\n                    vdf = runner.run_validation(verbose=False)\n                st.session_state["val_df"] = vdf\n                st.success(f"Validation done — {len(vdf)} records")\n\n        exp_df = st.session_state.get("exp_df")\n        if exp_df is not None and not exp_df.empty:\n            st.markdown("#### Train/Test Results")\n            num_cols = [c for c in ["hallucination_rate", "completeness", "faithfulness",\n                                     "context_relevance", "latency_ms"] if c in exp_df.columns]\n\n            # Chunk size plot\n            if "target_size" in exp_df.columns:\n                g1 = exp_df.groupby("target_size")[num_cols].mean().reset_index()\n                fig1 = px.bar(g1, x="target_size",\n                               y=["hallucination_rate", "completeness"],\n                               barmode="group",\n                               title="Impact of Chunk Size",\n                               template="plotly_dark",\n                               color_discrete_sequence=["#e05454", "#3ecf8e"])\n                fig1.update_layout(paper_bgcolor="#0f0f13", plot_bgcolor="#0f0f13")\n                st.plotly_chart(fig1, use_container_width=True)\n\n            # Top-k plot\n            if "top_k" in exp_df.columns:\n                g2 = exp_df.groupby("top_k")[num_cols].mean().reset_index()\n                fig2 = px.line(g2, x="top_k",\n                                y=["hallucination_rate", "completeness", "context_relevance"],\n                                markers=True,\n                                title="Impact of Top-k",\n                                template="plotly_dark",\n                                color_discrete_sequence=["#e05454", "#3ecf8e", "#4f6ef7"])\n                fig2.update_layout(paper_bgcolor="#0f0f13", plot_bgcolor="#0f0f13")\n                st.plotly_chart(fig2, use_container_width=True)\n\n            with st.expander("Full results table"):\n                st.dataframe(exp_df, use_container_width=True)\n\n        val_df = st.session_state.get("val_df")\n        if val_df is not None and not val_df.empty:\n            st.markdown("#### Validation Results (RAG paper)")\n            num_cols_v = [c for c in ["hallucination_rate", "completeness", "faithfulness"]\n                          if c in val_df.columns]\n            st.dataframe(val_df[["question", "verdict"] + num_cols_v]\n                         if "question" in val_df.columns else val_df,\n                         use_container_width=True)\n\n    # ── Tab 7: Design ─────────────────────────────────────────────────────────\n    with tabs[6]:\n        st.markdown("""\n### System Design & Justification\n\n#### Why RAG Instead of Fine-Tuning?\n| Aspect | RAG | Fine-Tuning |\n|--------|-----|-------------|\n| **Training data needed** | Only PDFs (no labelled Q&A) | Thousands of Q&A pairs |\n| **GPU cost** | Inference only | Multi-day training |\n| **Knowledge update** | Add new PDF → re-index | Full retrain |\n| **Grounding** | Explicit context → auditable | Implicit → opaque |\n| **Hallucination control** | Strict prompt enforces grounding | Hard to guarantee |\n\n#### Why all-MiniLM-L6-v2?\n- 384-d vectors → fast FAISS search even on CPU\n- 5× faster than BERT-large with 95% semantic quality\n- L2-normalised → cosine similarity via inner product (no extra compute)\n- Strong BEIR/MTEB retrieval benchmark scores\n\n#### Chunking Strategy Trade-offs\n| Strategy | Strengths | Weaknesses |\n|----------|-----------|------------|\n| Fixed | Predictable size | Cuts sentences mid-thought |\n| Sentence | Natural boundaries | Variable size |\n| Dynamic | Adapts to content flow | Slightly slower |\n| Overlapping | Cross-chunk context | Redundancy, larger index |\n| Heading | Preserves paper structure | Depends on formatting |\n\n**Recommendation:** Dynamic (200–800 tokens) is the best default —\nit respects sentence boundaries while staying within a useful token range.\n\n#### Accuracy vs Latency Trade-offs\n- **k=2** → fastest, may miss relevant chunks\n- **k=5** → best recall, more tokens → slower LLM\n- **Reranker** → +20% precision, +35% latency\n- **Strict prompt** → lower hallucination, occasionally "not specified"\n- **Open prompt** → more complete, higher hallucination risk\n\n#### Dataset Split (Critical)\n```\nTrain/Test : Attention Is All You Need (2017)\n             BERT (2018)\n             → Used for: chunking strategy selection, prompt tuning, k optimisation\n\nValidation : RAG paper (2020)\n             → Used ONLY for: final generalization evaluation\n             → Never used during system development\n```\n        """)\n\n\nif __name__ == "__main__":\n    main()\n')
print('  wrote app.py')

print('\n✅ All source files written to', PROJECT)


---
## Cell 3 — Verify imports

In [ ]:
import sys, os
sys.path.insert(0, '/content/rag_project')
os.chdir('/content/rag_project')

from dataset          import load_paper, PAPERS
from chunking         import get_chunks, compare_strategies, STRATEGIES
from retrieval        import RetrievalPipeline
from llm              import LLMEngine
from hallucination    import HallucinationDetector
from evaluation       import Evaluator
from pipeline         import RAGPipeline, build_pipeline
from experiments      import ExperimentRunner
from comparison       import ComparisonEngine
from question_generator import QuestionGenerator

print('✅ All modules imported')
print(f'   Papers    : {list(PAPERS.keys())}')
print(f'   Strategies: {list(STRATEGIES.keys())}')

---
## Cell GPU — Check GPU and RAM

In [ ]:
import torch, subprocess
print('-'*45)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  Device    : {device}')
if device == 'cuda':
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'  GPU       : {name}')
    print(f'  VRAM      : {vram:.1f} GB')
    print('  TinyLlama : SUPPORTED  (run Cell LLM to activate)')
else:
    print('  GPU       : none')
    print('  TinyLlama : not available  (mock LLM is instant and works fine)')
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f'  RAM total : {ram.total/1e9:.1f} GB')
    print(f'  RAM free  : {ram.available/1e9:.1f} GB')
except:
    r = subprocess.run(['free','-h'], capture_output=True, text=True)
    if r.returncode==0:
        [print(f'  {l}') for l in r.stdout.strip().split('\n')[:2]]
print('-'*45)

---
## Cell 4 — Configuration

> Change `LLM_BACKEND` to `"tinyllama"` if GPU is available. Keep `"mock"` otherwise.

In [ ]:
# LLM_BACKEND:
#   'mock'      -> instant, no GPU, works everywhere  <- default
#   'tinyllama' -> real LLM, needs T4 GPU (~2GB VRAM)
LLM_BACKEND    = 'mock'
CHUNK_STRATEGY = 'dynamic'
CHUNK_SIZE     = 500
TOP_K          = 5
USE_RERANKER   = False
PROMPT_STYLE   = 'strict'

import torch
print(f'LLM backend   : {LLM_BACKEND}')
print(f'Strategy      : {CHUNK_STRATEGY}  size={CHUNK_SIZE}')
print(f'Top-k         : {TOP_K}')
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available() and LLM_BACKEND == 'mock':
    print('  TIP: GPU detected! Set LLM_BACKEND = "tinyllama" for real LLM answers.')

---
## Cell 5 — Load Attention + BERT (Train/Test papers)

> RAG paper is NOT loaded here — it is the unseen validation set.

In [ ]:
import sys, os
sys.path.insert(0, '/content/rag_project')
os.chdir('/content/rag_project')
from pipeline import build_pipeline

pipeline = build_pipeline(
    backend=LLM_BACKEND,
    strategy=CHUNK_STRATEGY,
    chunk_size=CHUNK_SIZE,
    top_k=TOP_K,
    use_reranker=USE_RERANKER,
    prompt_style=PROMPT_STYLE,
    data_dir='/content/rag_project/data',
)

print('=== TRAIN/TEST PHASE ===')
print('Downloading + indexing (arXiv -> PDF -> chunks -> FAISS)...\n')
pipeline.load_paper('attention')
pipeline.load_paper('bert')
print(f'\nLoaded: {pipeline.loaded_papers}')
print(f'Attention chunks: {len(pipeline.get_chunks("attention"))}')
print(f'BERT chunks     : {len(pipeline.get_chunks("bert"))}')

---
## Cell 6 — Query Attention paper

In [ ]:
Q = 'What is the core innovation of the Transformer and how does multi-head attention work?'
print(f'Question: {Q}\n')
resp_attn = pipeline.query('attention', Q)
print(resp_attn.answer)
print('\n-- Hallucination Report --')
print(resp_attn.hall_report.summary())
print(f'\nLatency: {resp_attn.latency_ms:.0f}ms | Chunks: {len(resp_attn.retrieved)}')

---
## Cell 7 — Query BERT paper

In [ ]:
Q2 = 'What pre-training objectives does BERT use and why are they effective?'
print(f'Question: {Q2}\n')
resp_bert = pipeline.query('bert', Q2)
print(resp_bert.answer)
print('\n-- Hallucination Report --')
print(resp_bert.hall_report.summary())

---
## Cell 8 — Compare all 5 chunking strategies

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from chunking import compare_strategies
doc  = pipeline.get_doc('attention')
rows = compare_strategies(doc.full_text, 'attention', sizes=[200,500,800])
df_ck = pd.DataFrame([r for r in rows if 'error' not in r])
print('Chunking Strategy Comparison\n')
print(df_ck.to_string(index=False))
fig, axes = plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Chunking Strategy Comparison', fontweight='bold')
for ax,col,color in zip(axes,['num_chunks','avg_tokens','coverage'],['steelblue','darkorange','seagreen']):
    sub = df_ck[df_ck['target_size']==500]
    ax.bar(sub['strategy'], sub[col], color=color)
    ax.set_title(col.replace('_',' ').title())
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('/content/chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> /content/chunking_comparison.png')

---
## Cell 9 — Run all experiments (Train/Test only)

In [ ]:
from experiments import ExperimentRunner
runner = ExperimentRunner(backend=LLM_BACKEND, data_dir='/content/rag_project/data', use_reranker=USE_RERANKER)
print('Running experiments on Attention + BERT...')
df_exp = runner.run_train_test(verbose=True)
print('\n-- Summary --')
runner.print_summary()

---
## Cell 10 — Visualise experiment results

In [ ]:
import matplotlib.pyplot as plt
df = runner.results_df()
num_cols = [c for c in ['hallucination_rate','completeness','faithfulness','context_relevance','latency_ms'] if c in df.columns]
fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Experiment Results - Train/Test Phase', fontweight='bold')
if 'target_size' in df.columns:
    g1 = df.groupby('target_size')[num_cols].mean()
    g1[['hallucination_rate','completeness']].plot(kind='bar',ax=axes[0],title='Chunk Size',color=['tomato','steelblue'])
    axes[0].tick_params(axis='x',rotation=0)
if 'top_k' in df.columns:
    g2 = df.groupby('top_k')[num_cols].mean()
    cols2 = [c for c in ['context_relevance','hallucination_rate'] if c in g2.columns]
    if cols2: g2[cols2].plot(kind='line',ax=axes[1],marker='o',title='Top-k',color=['steelblue','tomato'])
if 'prompt_style' in df.columns:
    g3 = df.groupby('prompt_style')[num_cols].mean()
    g3[['hallucination_rate','completeness']].plot(kind='bar',ax=axes[2],title='Prompt Style',color=['tomato','steelblue'])
    axes[2].tick_params(axis='x',rotation=0)
plt.tight_layout()
plt.savefig('/content/experiment_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 11 — Critical research questions

In [ ]:
from question_generator import QuestionGenerator
gen = QuestionGenerator()
for pid in ['attention','bert']:
    qs = gen.generate(pid, n=5)
    print(qs.to_markdown())
    print('-'*60)

---
## Cell 12 — Multi-paper comparison

In [ ]:
from comparison import ComparisonEngine
import pandas as pd
table = ComparisonEngine().compare(['attention','bert'])
df_comp = pd.DataFrame(table.as_dict_list())
print('=== Attention vs BERT ===\n')
print(df_comp.to_string(index=False))
print(f'\nRecommendation:\n{table.recommendation}')

---
## Cell 13 — VALIDATION PHASE: RAG paper (unseen)

> First time this paper is ever loaded. Never used during development.

In [ ]:
print('=== VALIDATION PHASE ===')
print('Loading RAG paper for the first time...\n')
rag_doc = pipeline.load_validation_paper()
print(f'Loaded: {rag_doc}')
print(f'RAG chunks: {len(pipeline.get_chunks("rag"))}')

VAL_Q = [
    'How does RAG combine retrieval with generation?',
    'What retrieval mechanism does RAG use?',
    'What are the main limitations of RAG?',
    'How does RAG differ from standard seq2seq models?',
]
import pandas as pd
rows = []
print('\nValidation queries:\n')
for q in VAL_Q:
    r = pipeline.query('rag', q)
    rows.append({'question':q[:48]+'...','completeness':round(r.answer.completeness(),3),
                 'hall_score':r.hall_report.score,'verdict':r.hall_report.verdict})
    print(f'  [{r.hall_report.verdict:12s}]  hall={r.hall_report.score:.3f}  complete={r.answer.completeness():.2f}')
df_val = pd.DataFrame(rows)
print(f'\nAvg hallucination: {df_val["hall_score"].mean():.3f}')
print(f'Avg completeness : {df_val["completeness"].mean():.3f}')

---
## Cell 14 — Validation experiments on RAG paper

In [ ]:
df_val_exp = runner.run_validation(verbose=True)
num = [c for c in ['hallucination_rate','completeness','faithfulness'] if c in df_val_exp.columns]
if num:
    print('\nValidation Metrics (mean):')
    print(df_val_exp[num].mean().round(3).to_string())

---
## Cell 15 — RAG critical questions

In [ ]:
qs = QuestionGenerator().generate('rag', n=5)
print(qs.to_markdown())

---
## Cell 16 — Final results dashboard

In [ ]:
import pandas as pd
print('='*55)
print('  FINAL RESULTS DASHBOARD')
print('='*55)
print(f'  Backend  : {LLM_BACKEND}')
print(f'  Strategy : {CHUNK_STRATEGY}  size={CHUNK_SIZE}')
print(f'  Loaded   : {pipeline.loaded_papers}')
df_all = runner.results_df()
if not df_all.empty:
    num_cols = [c for c in ['hallucination_rate','completeness','faithfulness','context_relevance','latency_ms'] if c in df_all.columns]
    if 'exp_name' in df_all.columns:
        print('\nResults by experiment:')
        print(df_all.groupby('exp_name')[num_cols].mean().round(3).to_string())
    runner.save_results('/content/experiment_results.csv')
    print('\nCSV saved -> /content/experiment_results.csv')
print('\nSystem validated end-to-end')

---
## Cell UI — Launch Streamlit app (live public URL via ngrok)

**Steps:**
1. Get a FREE ngrok token at https://ngrok.com  
2. Go to https://dashboard.ngrok.com/get-started/your-authtoken  
3. Copy the token and paste it into `NGROK_TOKEN` below  
4. Run this cell — a public URL will appear  
5. Open the URL in any browser tab to see the full Streamlit app  

> The app runs on Colab's server (GPU if enabled). You just view it in your browser.

In [ ]:
import subprocess, sys, threading, time, os
sys.path.insert(0, '/content/rag_project')
os.chdir('/content/rag_project')

# ── METHOD 1: localtunnel (NO account or token needed) ────────────────────
# This is the easiest — just run this cell and get a URL instantly.
# If METHOD 1 doesn't work, use METHOD 2 below.

USE_LOCALTUNNEL = True   # set False to use ngrok instead

# ──────────────────────────────────────────────────────────────────────────

def kill_existing():
    subprocess.run(["pkill", "-f", "streamlit"],  capture_output=True)
    subprocess.run(["pkill", "-f", "lt"],          capture_output=True)
    subprocess.run(["pkill", "-f", "ngrok"],       capture_output=True)
    time.sleep(2)

def start_streamlit():
    """Start Streamlit on port 8501 in background."""
    subprocess.run([
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port",               "8501",
        "--server.headless",           "true",
        "--server.enableCORS",         "false",
        "--server.enableXsrfProtection","false",
        "--server.address",            "0.0.0.0",
    ])

kill_existing()

t = threading.Thread(target=start_streamlit, daemon=True)
t.start()
print("Streamlit starting on port 8501 ...")
time.sleep(5)

# ── METHOD 1: localtunnel (no signup, free, instant) ──────────────────────
if USE_LOCALTUNNEL:
    # Install localtunnel via npm (already on Colab)
    r = subprocess.run(["npm", "install", "-g", "localtunnel"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("npm install failed:", r.stderr[:200])
    else:
        # Get Colab's external IP for the password prompt localtunnel shows
        ip_result = subprocess.run(["curl", "-s", "https://api.ipify.org"],
                                   capture_output=True, text=True, timeout=10)
        colab_ip = ip_result.stdout.strip()

        # Start localtunnel tunnel
        lt_proc = subprocess.Popen(
            ["lt", "--port", "8501"],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        time.sleep(3)
        url_line = lt_proc.stdout.readline()
        url = url_line.strip().replace("your url is: ", "")

        print()
        print("=" * 55)
        print("  Streamlit app is LIVE!")
        print(f"  URL     :  {url}")
        print()
        print("  IMPORTANT — when the page loads:")
        print(f"  Enter this IP when prompted: {colab_ip}")
        print("  (localtunnel shows a password screen — enter the IP above)")
        print("=" * 55)
        print()
        print("  Tabs: Query & Answer | Hallucination | Chunks |")
        print("        Comparison | Questions | Experiments | Design")
        print()
        print("  To stop: Runtime -> Interrupt execution")

# ── METHOD 2: ngrok (needs free token from ngrok.com) ─────────────────────
else:
    NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
    # Get token at: https://dashboard.ngrok.com/get-started/your-authtoken
    # Paste ONLY the token — no spaces before or after

    token = NGROK_TOKEN.strip()   # removes accidental spaces/newlines

    if token == "PASTE_YOUR_NGROK_TOKEN_HERE" or len(token) < 10:
        print("ERROR: Set NGROK_TOKEN to your token from https://ngrok.com")
    else:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = token
        ngrok.kill()
        time.sleep(1)
        try:
            tunnel = ngrok.connect(8501, proto="http")
            url = tunnel.public_url
            print()
            print("=" * 55)
            print("  Streamlit app is LIVE!")
            print(f"  URL  :  {url}")
            print("=" * 55)
            print("  Open the URL — no password needed with ngrok.")
        except Exception as e:
            print(f"ngrok error: {e}")
            print()
            print("Token troubleshooting:")
            print("  1. Copy token from: https://dashboard.ngrok.com/get-started/your-authtoken")
            print("  2. Paste ONLY the token string — no extra spaces")
            print("  3. Token looks like: 2abc123_xyz456...")
            print("  Or set USE_LOCALTUNNEL = True above to skip ngrok entirely.")


---
## Cell LLM — Switch to TinyLlama (GPU only)

> Run this ONLY if you have a T4/A100 GPU. Then re-run Cell 5 to rebuild the pipeline.

In [ ]:
import torch
if not torch.cuda.is_available():
    print('No GPU detected.')
    print('Go to: Runtime -> Change runtime type -> T4 GPU')
    print('Then re-run from Cell 1.')
else:
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU: {gpu}  ({vram:.1f} GB VRAM)')
    print()
    # Update config variable
    LLM_BACKEND = 'tinyllama'
    print('LLM_BACKEND set to tinyllama')
    print()
    print('Now re-run Cell 5 to rebuild the pipeline with TinyLlama.')
    print('First run downloads the model (~1.1 GB) - takes ~2 min on T4.')
    print('All 7 app tabs will then use real LLM answers.')

---
## Quick reference

| Situation | Action |
|-----------|--------|
| Session disconnected | Re-run Cell 2, then Cell 5 |
| Want real LLM | Runtime -> T4 GPU -> Cell LLM -> Cell 5 |
| URL expired | Re-run Cell UI |
| Download CSV | Files panel -> `/content/experiment_results.csv` -> Download |
| Keep PDFs between sessions | Mount Drive, set `data_dir='/content/drive/MyDrive/rag_papers'` in Cell 5 |